# University Student Mental Stress Pattern Analysis - Unsupervised Learning

**KUET, CSE 4112 Machine Learning Laboratory**

Clustering 987 anonymous survey responses from Bangladeshi engineering students to
find interpretable *stress profiles*, with the free-text answers analysed through a
purpose-built code-mixed lexicon.

This notebook is generated from the project's `ml/` package, so what runs here is
exactly what runs locally. It is fully self-contained: **no pip install and no
internet are required** for the main pipeline.

## What it does

| Step | Method |
|---|---|
| Integrity audit | completeness, duplicates, range checks, straight-lining |
| Direction alignment | the two positively worded items recoded `6 - x` |
| Measurement structure | Cronbach's alpha, item-total, KMO, Bartlett |
| Dimensionality reduction | PCA + Horn's parallel analysis + varimax rotation |
| Clustering | k-means, Gaussian mixture (EM), Ward/average/complete hierarchical, spectral, Gower k-medoids |
| Choosing k | SSE elbow (Kneedle), silhouette, Davies-Bouldin, Calinski-Harabasz, BIC, CV log-likelihood, **gap statistic** |
| Validation | bootstrap ARI, seed stability, consensus matrix, classes-to-clusters against held-out demographics |
| Free text | frozen 13-theme code-mixed lexicon, NMF + LDA cross-check, two pre-specified null tests |
| Profiling | z-score profiles, data-derived names, eta-squared item ranking, recommendations |
| Supervised | profile recovery and the incremental value of text features, vs an explicit baseline |

## Setup (one manual step)

Attach the response workbook as a Kaggle dataset via **+ Add Input**. The loader
finds any file matching `*Academic Stress among Bangladeshi Engineering Students*.xlsx`
anywhere under `/kaggle/input`.

GPU is **not** needed. The optional embedding cross-check in the last section is
the only part that wants a GPU and internet.

## 0. Environment

In [ ]:
import os, sys, platform, warnings
warnings.filterwarnings("ignore")

print("python  ", platform.python_version())
for m in ("numpy", "pandas", "scipy", "sklearn", "matplotlib", "joblib"):
    try:
        print("%-11s %s" % (m, __import__(m).__version__))
    except Exception as e:
        print("%-11s MISSING (%s)" % (m, e))

if os.path.isdir("/kaggle/input"):
    print("\nAttached inputs:")
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            print("   ", os.path.join(root, f))
else:
    print("\nNot on Kaggle - running against the local project folder.")

## 1. The pipeline package

The next cells write the `ml/` package to disk and import it. Each cell is one
module; the docstring at the top of each explains the choices it makes and why.

In [ ]:
import os, sys

# The module cells below use `%writefile ml/<name>.py`, which resolves against the
# working directory. On Kaggle that is /kaggle/working, so nothing outside the
# session is touched. If you run this notebook locally, run it from a scratch
# directory - otherwise it rewrites the project's own ml/ package with this snapshot.
os.makedirs("ml", exist_ok=True)
with open(os.path.join("ml", "__init__.py"), "w", encoding="utf-8") as fh:
    fh.write('__version__ = "1.0.0"\n')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("writing package into:", os.path.abspath("ml"))

### `ml/config.py` - Single source of truth for paths, column mapping, item metadata and plot tokens.

In [ ]:
%%writefile ml/config.py
# -*- coding: utf-8 -*-
"""
Single source of truth for paths, column mapping, item metadata and plot tokens.

Every other module imports from here, so the pipeline behaves identically on a
local Windows machine and inside a Kaggle kernel: the only thing that changes is
where `resolve_data_path()` finds the workbook and where OUT_DIR points.
"""
from __future__ import annotations

import glob
import os

# --------------------------------------------------------------------------
# Environment detection
# --------------------------------------------------------------------------
ON_KAGGLE = os.path.isdir("/kaggle/input")

PKG_DIR = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(PKG_DIR)

#: Filenames this project has used for the same export, newest first. The
#: pipeline accepts any of them so a re-download from Google Forms (which always
#: lands as "... (Responses).xlsx") does not break the run.
DATA_FILENAMES = [
    "Academic Stress among Bangladeshi Engineering Students (Responses).xlsx",
    "Academic Stress among Bangladeshi Engineering Students (Finalv2).xlsx",
    "Academic Stress among Bangladeshi Engineering Students (Final).xlsx",
]
DATA_GLOB = "*Academic Stress among Bangladeshi Engineering Students*.xlsx"


def resolve_data_path(explicit: str | None = None) -> str:
    """Locate the response workbook.

    Search order: an explicit path / the STRESS_DATA env var -> the Kaggle input
    mounts -> the repository root. Raises with an actionable message rather than
    letting pandas fail on a path the user cannot see.
    """
    explicit = explicit or os.environ.get("STRESS_DATA")
    if explicit:
        if os.path.isfile(explicit):
            return explicit
        raise FileNotFoundError(f"Explicit data path does not exist: {explicit}")

    roots = []
    if ON_KAGGLE:
        roots += sorted(glob.glob("/kaggle/input/*"))
        roots.append("/kaggle/working")
    roots.append(ROOT)

    for base in roots:
        for name in DATA_FILENAMES:
            cand = os.path.join(base, name)
            if os.path.isfile(cand):
                return cand
        hits = sorted(glob.glob(os.path.join(base, DATA_GLOB)))
        if hits:
            return hits[0]

    raise FileNotFoundError(
        "Could not find the response workbook.\n"
        f"Looked for {DATA_FILENAMES[0]!r} (or {DATA_GLOB!r}) in: {roots}\n"
        "On Kaggle: attach the dataset via '+ Add Input'. Locally: set STRESS_DATA."
    )


OUT_DIR = "/kaggle/working" if ON_KAGGLE else os.path.join(ROOT, "outputs")
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
MODEL_DIR = os.path.join(OUT_DIR, "models")


def ensure_dirs() -> None:
    for d in (OUT_DIR, FIG_DIR, TAB_DIR, MODEL_DIR):
        os.makedirs(d, exist_ok=True)


# --------------------------------------------------------------------------
# Column mapping
# --------------------------------------------------------------------------
# The Google Forms export carries the full bilingual question text as the column
# header, so columns are addressed positionally and then verified against a
# keyword fingerprint (see dataio.validate_schema).
COL_IDX = {
    "timestamp": 0,
    "year": 1,
    "cgpa": 2,
    "gender": 3,
    "living": 4,
    "backlog": 11,
    "open_current": 18,
    "open_previous": 19,
    "department": 20,
}
LIKERT_IDX = [5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17]

#: Short keys for the twelve Likert items, in questionnaire order.
ITEMS = [
    "MissMeal", "PileUp", "SleepLoss", "LabStress", "ExamWorry", "ResultDemotiv",
    "CGPACompare", "AskTeacher", "Feedback", "Financial", "JobWorry", "SocioPol",
]

#: Q-numbers used in the dataset-characteristics report, for cross-referencing.
ITEM_QNUM = {it: f"Q{i + 1}" for i, it in enumerate(ITEMS)}

ITEM_LABEL = {
    "MissMeal": "Miss meals because of classes",
    "PileUp": "Assignments / lab reports pile up",
    "SleepLoss": "Sacrifice sleep for coursework",
    "LabStress": "Lab work more stressful than theory",
    "ExamWorry": "Worry about results despite preparing",
    "ResultDemotiv": "One poor result kills motivation",
    "CGPACompare": "Constantly compare CGPA with peers",
    "AskTeacher": "Uncomfortable asking teachers for help (R)",
    "Feedback": "Instructor feedback does not reduce stress (R)",
    "Financial": "Financial concerns hurt performance",
    "JobWorry": "Worry about a job after graduation",
    "SocioPol": "Socio-economic / political instability",
}

#: Positively worded items. Recoded as 6 - x so that HIGH always means MORE
#: strain / LESS support across the whole instrument (methodology report S5.2).
REVERSED = ["AskTeacher", "Feedback"]

#: One-word fingerprints used to assert the positional mapping still holds.
SCHEMA_FINGERPRINT = {
    1: "year", 2: "cgpa", 3: "gender", 4: "living",
    5: "breakfast", 6: "pile", 7: "sleep", 8: "lab", 9: "poor result",
    10: "motivation", 11: "backlog", 12: "compare", 13: "comfortable",
    14: "feedback", 15: "financial", 16: "job", 17: "instability",
    18: "biggest source", 19: "previous academic", 20: "department",
}

# --------------------------------------------------------------------------
# Category level orders (fixed so every table/figure sorts identically)
# --------------------------------------------------------------------------
YEAR_ORD = ["1st Year", "2nd Year", "3rd Year", "4th Year", "Other"]
CGPA_ORD = ["Below 2.50", "2.50\u20132.99", "3.00\u20133.49", "3.50\u20133.79", "3.80\u20134.00"]
GENDER_ORD = ["Male", "Female"]
LIVE_ORD = ["Hall", "Mess", "Family", "Other"]

#: Ordinal encodings for the two ordered background variables.
YEAR_ORDINAL = {"1st Year": 1, "2nd Year": 2, "3rd Year": 3, "4th Year": 4, "Other": 0}
CGPA_ORDINAL = {c: i + 1 for i, c in enumerate(CGPA_ORD)}

# --------------------------------------------------------------------------
# Modelling constants
# --------------------------------------------------------------------------
RANDOM_STATE = 42
K_RANGE = list(range(2, 9))          # k = 2..8, as specified in the methodology
N_INIT = 50                          # k-means restarts
PCA_VAR_TARGET = 0.95                # retain components covering 95% of variance
BOOTSTRAP_B = 200                    # resamples for the ARI stability check
SEED_TRIALS = 20                     # distinct seeds for the seed-stability check
CV_FOLDS = 5

# --------------------------------------------------------------------------
# Design tokens (validated light-surface palette, print-safe)
# --------------------------------------------------------------------------
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BASE = "#c3c2b7"
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
DIV5 = ["#b02f2f", "#e88a89", "#cfcec7", "#86b6ef", "#1c5cab"]
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

### `ml/dataio.py` - Loading, schema validation, integrity auditing and export.

In [ ]:
%%writefile ml/dataio.py
# -*- coding: utf-8 -*-
"""
Loading, schema validation, integrity auditing and export.

Nothing here silently repairs the data. Every check writes a number into the
audit dictionary so the report can state what was found (including "nothing"),
which is what the dataset-documentation sheet required by the course asks for.
"""
from __future__ import annotations

import json
import os
import re

import numpy as np
import pandas as pd

from . import config as C


# --------------------------------------------------------------------------
# Load
# --------------------------------------------------------------------------
def load_raw(path=None):
    """Read the workbook and return (dataframe, resolved_path)."""
    resolved = C.resolve_data_path(path)
    df = pd.read_excel(resolved)
    return df, resolved


def validate_schema(df):
    """Assert the positional column mapping still matches the header text.

    The export is addressed by position because the headers are long bilingual
    sentences. That is fragile if the form is ever edited, so each mapped index
    is fingerprinted against a keyword; a mismatch aborts the run instead of
    producing a plausible-looking but wrong analysis.
    """
    cols = list(df.columns)
    if len(cols) != 21:
        raise ValueError("Expected 21 columns in the export, found %d." % len(cols))

    problems = []
    for idx, cue in C.SCHEMA_FINGERPRINT.items():
        if cue.lower() not in str(cols[idx]).lower():
            problems.append("column %d (%r) does not contain %r" % (idx, str(cols[idx])[:60], cue))
    if problems:
        raise ValueError("Column mapping no longer matches the export:\n  " + "\n  ".join(problems))

    return {"n_columns": len(cols), "columns": [str(c) for c in cols]}


def column_names(df):
    """Map the logical names in config.COL_IDX onto real column labels."""
    cols = list(df.columns)
    names = {k: cols[i] for k, i in C.COL_IDX.items()}
    names["likert"] = [cols[i] for i in C.LIKERT_IDX]
    return names


# --------------------------------------------------------------------------
# Integrity audit
# --------------------------------------------------------------------------
def audit(df, names):
    """Completeness, range and duplication checks over the raw export."""
    lik = df[names["likert"]]
    closed = [names[k] for k in ("year", "cgpa", "gender", "living", "backlog", "department")]

    out_of_range = int(((lik < 1) | (lik > 5)).to_numpy().sum())
    non_integer = int((lik != lik.round()).to_numpy().sum())

    ts = pd.to_datetime(df[names["timestamp"]], errors="coerce")
    by_day = ts.dt.date.value_counts()

    return {
        "n_rows": int(len(df)),
        "closed_ended_missing": int(df[closed].isna().to_numpy().sum() + lik.isna().to_numpy().sum()),
        "open_current_missing": int(df[names["open_current"]].isna().sum()),
        "open_previous_missing": int(df[names["open_previous"]].isna().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "duplicate_ignoring_timestamp": int(df.drop(columns=[names["timestamp"]]).duplicated().sum()),
        "likert_out_of_range": out_of_range,
        "likert_non_integer": non_integer,
        "collection_start": str(ts.min()),
        "collection_end": str(ts.max()),
        "collection_days": int((ts.max() - ts.min()).days) + 1,
        "peak_day": str(by_day.idxmax()),
        "peak_day_n": int(by_day.max()),
    }


def response_style(items):
    """Straight-lining / acquiescence diagnostics on the direction-aligned block."""
    within_sd = items.std(axis=1)
    same_all = int((items.nunique(axis=1) == 1).sum())

    def longest_run(row):
        best = run = 1
        for a, b in zip(row[:-1], row[1:]):
            run = run + 1 if a == b else 1
            best = max(best, run)
        return best

    runs = items.apply(lambda r: longest_run(list(r)), axis=1)
    n = len(items)
    return {
        "straight_lining_n": same_all,
        "straight_lining_pct": round(100 * same_all / n, 2),
        "low_variance_n": int((within_sd < 0.35).sum()),
        "low_variance_pct": round(100 * float((within_sd < 0.35).mean()), 2),
        "mean_within_row_sd": round(float(within_sd.mean()), 3),
        "long_run_ge8_n": int((runs >= 8).sum()),
        "long_run_ge8_pct": round(100 * float((runs >= 8).mean()), 2),
        "acquiescence_pct_top_box": round(100 * float((items == 5).to_numpy().mean()), 1),
        "pct_using_all_five_points": round(100 * float((items.nunique(axis=1) == 5).mean()), 1),
    }


# --------------------------------------------------------------------------
# Export
# --------------------------------------------------------------------------
_ARFF_SAFE = re.compile(r"[^0-9A-Za-z_]+")


def _arff_name(s):
    return _ARFF_SAFE.sub("_", str(s)).strip("_")


def _arff_value(v):
    if pd.isna(v):
        return "?"
    s = str(v)
    s = s.replace(chr(92), chr(92) * 2)
    s = s.replace("'", chr(92) + "'")
    s = s.replace("\n", " ").replace("\r", " ")
    return "'" + s + "'"


def write_arff(df, path, relation, numeric, nominal, string_cols):
    """Write an ARFF file so the same prepared table can be opened in WEKA.

    The methodology report specifies a WEKA workflow; this keeps that route open
    from the identical preprocessing the Python pipeline uses, instead of
    preparing the data twice and hoping the two agree.
    """
    lines = ["@RELATION " + _arff_name(relation), ""]
    order = []
    for c in df.columns:
        nm = _arff_name(c)
        if c in numeric:
            lines.append("@ATTRIBUTE %s NUMERIC" % nm)
        elif c in nominal:
            levels = ",".join(_arff_value(v) for v in nominal[c])
            lines.append("@ATTRIBUTE %s {%s}" % (nm, levels))
        elif c in string_cols:
            lines.append("@ATTRIBUTE %s STRING" % nm)
        else:
            continue
        order.append(c)
    lines += ["", "@DATA"]
    for _, row in df[order].iterrows():
        vals = []
        for c in order:
            v = row[c]
            if c in numeric:
                vals.append("?" if pd.isna(v) else ("%g" % float(v)))
            else:
                vals.append(_arff_value(v))
        lines.append(",".join(vals))

    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")
    return path


def jsonable(obj):
    """Recursively convert numpy scalars/arrays so json.dump never chokes."""
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return jsonable(obj.tolist())
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, float) and np.isnan(obj):
        return None
    if isinstance(obj, pd.Timestamp):
        return str(obj)
    return obj


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(jsonable(obj), fh, indent=2, ensure_ascii=False)
    return path


def save_table(df, name, index=True):
    """Write a results table to outputs/tables as UTF-8-BOM CSV (Excel-friendly)."""
    path = os.path.join(C.TAB_DIR, name + ".csv")
    df.to_csv(path, index=index, encoding="utf-8-sig")
    return path

### `ml/preprocess.py` - Preprocessing: direction alignment, encoding and the standardised feature matrix.

In [ ]:
%%writefile ml/preprocess.py
# -*- coding: utf-8 -*-
"""
Preprocessing: direction alignment, encoding and the standardised feature matrix.

Implements section 5.2 of the methodology report:
  * the two positively worded items are recoded 6 - x, so HIGH is always
    "more strain / less support" for all twelve items;
  * ordered background fields become ordinal integers, unordered ones one-hot;
  * clustering features are z-scored, because the item SDs range 0.97-1.46 and
    un-standardised Euclidean distance would let the two most polarised items
    dominate every centroid.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from . import config as C


def build_items(df, names):
    """Return the 12 Likert items, direction-aligned, with short column names.

    Raw (un-recoded) values are returned alongside because response-composition
    figures and the data sheet must show what students actually ticked.
    """
    raw = df[names["likert"]].copy()
    raw.columns = C.ITEMS
    raw = raw.astype(int)

    items = raw.copy()
    for c in C.REVERSED:
        items[c] = 6 - items[c]
    return items, raw


def build_background(df, names):
    """Assemble the background block with explicit ordinal / nominal treatment."""
    bg = pd.DataFrame(index=df.index)
    bg["year"] = df[names["year"]].astype(str)
    bg["cgpa"] = df[names["cgpa"]].astype(str)
    bg["gender"] = df[names["gender"]].astype(str)
    bg["living"] = df[names["living"]].astype(str)
    bg["department"] = df[names["department"]].astype(str)
    # The backlog item ships as a bilingual "Yes / হ্যাঁ" string; match on the
    # English stem so a change to the Bangla half cannot flip the encoding.
    bg["backlog"] = df[names["backlog"]].astype(str).str.strip().str.lower().str.startswith("yes").astype(int)

    bg["year_ord"] = bg["year"].map(C.YEAR_ORDINAL).fillna(0).astype(int)
    bg["cgpa_ord"] = bg["cgpa"].map(C.CGPA_ORDINAL).fillna(0).astype(int)
    return bg


def encode_background(bg, include_department=True, exclude=()):
    """One-hot the nominal background fields, keep the ordinals as integers.

    Used only for the mixed-data cross-check and the supervised extension. The
    primary clustering runs on the twelve items alone (methodology 5.1), so the
    background variables stay external and can serve as held-out validation.

    `exclude` drops encoded columns by name. It exists because a supervised run
    that predicts one background variable must not be handed that same variable
    as a feature: leaving `cgpa_ord` in while predicting the CGPA band produces
    a 94%-accurate model that has learned nothing.
    """
    nominal = ["gender", "living"] + (["department"] if include_department else [])
    enc = pd.get_dummies(bg[nominal], prefix=nominal, drop_first=False).astype(int)
    enc["year_ord"] = bg["year_ord"]
    enc["cgpa_ord"] = bg["cgpa_ord"]
    enc["backlog"] = bg["backlog"]

    drop = [c for c in enc.columns
            if c in exclude or any(c.startswith(p + "_") for p in exclude)]
    return enc.drop(columns=drop)


def standardise(items):
    """Z-score the item block; returns (Z, fitted_scaler)."""
    scaler = StandardScaler()
    Z = scaler.fit_transform(items.astype(float).to_numpy())
    return Z, scaler


def composite_score(items):
    """Mean of the twelve aligned items.

    Reported as a descriptive summary only. Section 3.4 of the dataset report
    established alpha = 0.61, which is below the threshold for treating this as
    a single measured construct, so it is never used as a modelling target.
    """
    return items.mean(axis=1)


def distribution(series, order=None):
    """Counts and percentages for one categorical column, in a fixed level order."""
    n = len(series)
    vc = series.value_counts()
    if order:
        keep = [o for o in order if o in vc.index]
        extra = [i for i in vc.index if i not in order]
        vc = vc.reindex(keep + extra)
    return pd.DataFrame({"n": vc, "pct": (100 * vc / n).round(1)})


def imbalance_report(bg):
    """Quantify the four sample skews the methodology report flags as limits."""
    g = bg["gender"].value_counts()
    d = bg["department"].value_counts()
    y = bg["year"].value_counts()
    n = len(bg)
    return {
        "gender_ratio": round(float(g.max() / max(g.min(), 1)), 2),
        "dept_max_min_ratio": round(float(d.max() / max(d.min(), 1)), 1),
        "dept_top3_pct": round(float(100 * d.nlargest(3).sum() / n), 1),
        "dept_below_25": int((d < 25).sum()),
        "backlog_minority_pct": round(float(100 * bg["backlog"].mean()), 1),
        "lower_years_pct": round(float(100 * (y.get("1st Year", 0) + y.get("2nd Year", 0)) / n), 1),
    }


def item_descriptives(items, raw):
    """Per-item table: raw distribution plus direction-aligned mean/SD/skew."""
    from scipy import stats

    rows = []
    for it in C.ITEMS:
        r = raw[it]
        a = items[it]
        counts = r.value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
        rows.append({
            "item": it,
            "q": C.ITEM_QNUM[it],
            "label": C.ITEM_LABEL[it],
            "positively_worded": it in C.REVERSED,
            "raw_mean": round(float(r.mean()), 3),
            "raw_sd": round(float(r.std()), 3),
            "raw_median": float(r.median()),
            "raw_skew": round(float(stats.skew(r)), 3),
            "pct_agree_raw": round(float(100 * (r >= 4).mean()), 1),
            "pct_disagree_raw": round(float(100 * (r <= 2).mean()), 1),
            "aligned_mean": round(float(a.mean()), 3),
            "aligned_sd": round(float(a.std()), 3),
            **{"n_%d" % v: int(counts[v]) for v in range(1, 6)},
        })
    return pd.DataFrame(rows).set_index("item")

### `ml/structure.py` - Measurement structure of the questionnaire.

In [ ]:
%%writefile ml/structure.py
# -*- coding: utf-8 -*-
"""
Measurement structure of the questionnaire.

This runs *before* any clustering on purpose. If the twelve items do not hang
together, a single stress score is not a valid target and the clustering result
has to be read as "profiles of response patterns", not "levels of one latent
stress trait". Establishing that first is what keeps the interpretation honest.

Provides: Cronbach's alpha (with alpha-if-item-deleted), corrected item-total
correlations, the inter-item correlation matrix, and the two standard
factorability tests (Bartlett's sphericity, Kaiser-Meyer-Olkin).
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

from . import config as C


def cronbach_alpha(frame):
    """Standard raw-score Cronbach's alpha."""
    k = frame.shape[1]
    if k < 2:
        return float("nan")
    item_var = frame.var(axis=0, ddof=1).sum()
    total_var = frame.sum(axis=1).var(ddof=1)
    return float((k / (k - 1)) * (1 - item_var / total_var))


def corrected_item_total(frame):
    """Correlation of each item with the sum of the *other* items."""
    total = frame.sum(axis=1)
    return {c: float(np.corrcoef(frame[c], total - frame[c])[0, 1]) for c in frame.columns}


def alpha_if_deleted(frame):
    """Alpha recomputed with each item dropped in turn."""
    out = {}
    for c in frame.columns:
        out[c] = round(cronbach_alpha(frame.drop(columns=[c])), 3)
    return out


def bartlett_sphericity(frame):
    """Bartlett's test that the correlation matrix is not an identity matrix.

    A non-significant result would mean the items are mutually uncorrelated and
    no factor/PCA solution is meaningful.
    """
    R = np.corrcoef(frame.to_numpy(), rowvar=False)
    n, p = frame.shape
    det = np.linalg.det(R)
    det = max(det, 1e-12)
    chi2 = -(n - 1 - (2 * p + 5) / 6) * np.log(det)
    dof = p * (p - 1) / 2
    return {"chi2": round(float(chi2), 2), "dof": int(dof),
            "p": float(stats.chi2.sf(chi2, dof)), "determinant": float(det)}


def kmo(frame):
    """Kaiser-Meyer-Olkin sampling adequacy, overall and per item.

    KMO compares correlations against partial correlations. Values below ~0.60
    say the items share too little common variance for a factor solution to be
    trustworthy - a number worth reporting either way.
    """
    R = np.corrcoef(frame.to_numpy(), rowvar=False)
    inv = np.linalg.pinv(R)
    d = np.sqrt(np.diag(inv))
    partial = -inv / np.outer(d, d)
    np.fill_diagonal(partial, 0.0)
    Rc = R.copy()
    np.fill_diagonal(Rc, 0.0)

    r2 = (Rc ** 2).sum()
    p2 = (partial ** 2).sum()
    overall = r2 / (r2 + p2)

    per = {}
    for i, c in enumerate(frame.columns):
        ri = (Rc[i] ** 2).sum()
        pi = (partial[i] ** 2).sum()
        per[c] = round(float(ri / (ri + pi)), 3)
    return {"overall": round(float(overall), 3), "per_item": per}


def analyse(items):
    """Full measurement-structure report over the direction-aligned items."""
    frame = items[C.ITEMS].astype(float)
    R = frame.corr()
    offdiag = R.to_numpy()[~np.eye(len(C.ITEMS), dtype=bool)]

    it_total = corrected_item_total(frame)
    weak = sorted([k for k, v in it_total.items() if v < 0.20], key=lambda k: it_total[k])
    core = [c for c in C.ITEMS if c not in weak]

    pairs = []
    for i, a in enumerate(C.ITEMS):
        for b in C.ITEMS[i + 1:]:
            pairs.append({"a": a, "b": b, "r": round(float(R.loc[a, b]), 3)})
    pairs.sort(key=lambda d: -abs(d["r"]))

    return {
        "cronbach_alpha_12": round(cronbach_alpha(frame), 3),
        "cronbach_alpha_core": round(cronbach_alpha(frame[core]), 3),
        "core_items": core,
        "weak_items": weak,
        "alpha_if_deleted": alpha_if_deleted(frame),
        "corrected_item_total": {k: round(v, 3) for k, v in it_total.items()},
        "mean_abs_interitem_r": round(float(np.abs(offdiag).mean()), 3),
        "mean_interitem_r": round(float(offdiag.mean()), 3),
        "max_interitem_r": round(float(offdiag.max()), 3),
        "min_interitem_r": round(float(offdiag.min()), 3),
        "top_pairs": pairs[:8],
        "bartlett": bartlett_sphericity(frame),
        "kmo": kmo(frame),
        "correlation_matrix": {a: {b: round(float(R.loc[a, b]), 3) for b in C.ITEMS} for a in C.ITEMS},
    }


def verdict(struct):
    """One-sentence, threshold-based reading of the structure numbers.

    Written as an explicit rule rather than prose so the conclusion cannot drift
    from the numbers it is based on.
    """
    a = struct["cronbach_alpha_12"]
    k = struct["kmo"]["overall"]
    r = struct["mean_abs_interitem_r"]
    if a >= 0.70 and k >= 0.70:
        return ("The twelve items behave as one reasonably reliable scale "
                "(alpha=%.2f, KMO=%.2f); a composite stress score is defensible." % (a, k))
    return ("The twelve items do not form a single reliable scale "
            "(alpha=%.2f, KMO=%.2f, mean |inter-item r|=%.2f). They are a checklist of "
            "partly independent stressors, so the analysis reports profiles across items "
            "rather than one stress level, and no composite score is used as a target." % (a, k, r))

### `ml/reduce.py` - Dimensionality reduction (methodology report section 5.4).

In [ ]:
%%writefile ml/reduce.py
# -*- coding: utf-8 -*-
"""
Dimensionality reduction (methodology report section 5.4).

PCA on the standardised item matrix, reported three ways so the reduction is
justified rather than assumed:
  * the full eigenvalue / explained-variance table with the Kaiser count and
    the number of components needed to reach the 95% variance target;
  * a parallel analysis against random data, which is a stricter and more
    honest component-retention rule than Kaiser on a 12-item instrument;
  * varimax-rotated loadings, because unrotated components on a low-alpha
    instrument are usually a size factor plus uninterpretable contrasts.

t-SNE is included only as a second projection for the cluster-overlap figure;
nothing is ever clustered in t-SNE space, since its distances are not metric.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from . import config as C


def varimax(loadings, gamma=1.0, max_iter=100, tol=1e-6):
    """Kaiser-normalised varimax rotation of a loading matrix."""
    L = np.asarray(loadings, dtype=float).copy()
    p, k = L.shape
    if k < 2:
        return L
    R = np.eye(k)
    d = 0.0
    for _ in range(max_iter):
        d_old = d
        Lam = L @ R
        u, s, vh = np.linalg.svd(
            L.T @ (Lam ** 3 - (gamma / p) * Lam @ np.diag(np.diag(Lam.T @ Lam)))
        )
        R = u @ vh
        d = float(s.sum())
        if d_old != 0 and abs(d - d_old) / d < tol:
            break
    return L @ R


def parallel_analysis(Z, n_iter=500, percentile=95, random_state=C.RANDOM_STATE):
    """Horn's parallel analysis: keep components beating random-data eigenvalues.

    Kaiser's eigenvalue>1 rule over-retains on short instruments. Comparing each
    observed eigenvalue against the 95th percentile of eigenvalues from
    same-shaped random normal data gives a defensible retention count.
    """
    rng = np.random.default_rng(random_state)
    n, p = Z.shape
    sim = np.empty((n_iter, p))
    for i in range(n_iter):
        X = rng.standard_normal((n, p))
        X = (X - X.mean(0)) / X.std(0, ddof=1)
        sim[i] = np.linalg.eigvalsh(np.corrcoef(X, rowvar=False))[::-1]
    threshold = np.percentile(sim, percentile, axis=0)
    observed = np.linalg.eigvalsh(np.corrcoef(Z, rowvar=False))[::-1]
    keep = int((observed > threshold).sum())
    return {
        "observed_eigenvalues": [round(float(v), 3) for v in observed],
        "random_p%d" % percentile: [round(float(v), 3) for v in threshold],
        "n_retain": keep,
    }


def run_pca(Z, item_names=None, var_target=C.PCA_VAR_TARGET):
    """Fit PCA on the standardised matrix and return (scores, model, report)."""
    item_names = list(item_names or C.ITEMS)
    full = PCA(random_state=C.RANDOM_STATE).fit(Z)
    eig = full.explained_variance_
    ratio = full.explained_variance_ratio_
    cum = np.cumsum(ratio)

    n_kaiser = int((eig > 1).sum())
    n_95 = int(np.searchsorted(cum, var_target) + 1)
    par = parallel_analysis(Z)

    scores = full.transform(Z)
    n_rot = max(2, par["n_retain"])
    raw_load = full.components_[:n_rot].T * np.sqrt(eig[:n_rot])
    rot_load = varimax(raw_load)

    load_df = pd.DataFrame(raw_load, index=item_names,
                           columns=["PC%d" % (i + 1) for i in range(n_rot)])
    rot_df = pd.DataFrame(rot_load, index=item_names,
                          columns=["RC%d" % (i + 1) for i in range(n_rot)])

    report = {
        "n_components_total": int(len(eig)),
        "eigenvalues": [round(float(v), 4) for v in eig],
        "explained_variance_pct": [round(float(v) * 100, 2) for v in ratio],
        "cumulative_variance_pct": [round(float(v) * 100, 2) for v in cum],
        "n_kaiser": n_kaiser,
        "n_for_%d_pct" % int(var_target * 100): n_95,
        "variance_at_2_components_pct": round(float(cum[1] * 100), 2),
        "parallel_analysis": par,
        "n_retained": n_rot,
        "loadings_unrotated": {c: {i: round(float(load_df.loc[i, c]), 3) for i in item_names}
                               for c in load_df.columns},
        "loadings_varimax": {c: {i: round(float(rot_df.loc[i, c]), 3) for i in item_names}
                             for c in rot_df.columns},
        "rotated_variance_pct": [round(float((rot_load[:, j] ** 2).sum() / len(item_names) * 100), 2)
                                 for j in range(n_rot)],
    }
    return scores, full, report, load_df, rot_df


def interpret_components(rot_df, threshold=0.40):
    """Name each rotated component by the items that load on it."""
    out = {}
    for c in rot_df.columns:
        s = rot_df[c]
        pos = [(i, round(float(v), 2)) for i, v in s.items() if v >= threshold]
        neg = [(i, round(float(v), 2)) for i, v in s.items() if v <= -threshold]
        pos.sort(key=lambda t: -t[1])
        neg.sort(key=lambda t: t[1])
        out[c] = {"positive": pos, "negative": neg,
                  "n_salient": len(pos) + len(neg)}
    return out


def tsne_projection(Z, random_state=C.RANDOM_STATE, perplexity=30):
    """2-D t-SNE embedding, used for visual overlap inspection only."""
    ts = TSNE(n_components=2, random_state=random_state, perplexity=perplexity,
              init="pca", max_iter=1000)
    return ts.fit_transform(Z)

### `ml/cluster.py` - Candidate clustering models and the k-selection evidence (methodology 5.5-5.6).

In [ ]:
%%writefile ml/cluster.py
# -*- coding: utf-8 -*-
"""
Candidate clustering models and the k-selection evidence (methodology 5.5-5.6).

Cluster count is decided from several independent signals rather than one:
  * within-cluster SSE across k = 2..8, read as an elbow (with the Kneedle
    knee-point computed numerically so "the elbow" is not eyeballed);
  * Silhouette, Davies-Bouldin and Calinski-Harabasz internal indices;
  * Gaussian-mixture BIC and cross-validated log-likelihood, which is the
    scikit-learn equivalent of WEKA's EM with -N -1 (automatic k);
  * Ward dendrogram structure.

Four algorithm families are fitted so the choice of algorithm is also evidenced:
k-means (primary), Gaussian mixture, Ward agglomerative, and a Gower-distance
k-medoids run over the mixed item+background matrix. The Gower branch is a
dependency-free implementation, so nothing here needs a package Kaggle does not
already ship.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import cophenet, dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.metrics import (calinski_harabasz_score, davies_bouldin_score,
                             silhouette_samples, silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import KFold

from . import config as C


# --------------------------------------------------------------------------
# Internal validity indices
# --------------------------------------------------------------------------
def internal_indices(Z, labels):
    """Silhouette / Davies-Bouldin / Calinski-Harabasz for one labelling."""
    uniq = len(set(labels))
    if uniq < 2 or uniq >= len(labels):
        return {"silhouette": None, "davies_bouldin": None, "calinski_harabasz": None}
    return {
        "silhouette": round(float(silhouette_score(Z, labels)), 4),
        "davies_bouldin": round(float(davies_bouldin_score(Z, labels)), 4),
        "calinski_harabasz": round(float(calinski_harabasz_score(Z, labels)), 2),
    }


def gap_statistic(Z, ks=None, n_refs=25, random_state=C.RANDOM_STATE):
    """Tibshirani's gap statistic, evaluated from k = 1 upward.

    This is the only criterion here that can answer "is there any structure at
    all", because it includes k = 1 as a candidate. Silhouette, Davies-Bouldin
    and Calinski-Harabasz are all undefined at k = 1 and therefore *cannot*
    report the absence of clusters - they only ever rank the partitions you ask
    for. Reference data is drawn uniformly over the bounding box of the
    principal components, which is the standard null of "one homogeneous blob".

    The selection rule is Tibshirani's: the smallest k whose gap is within one
    standard error of the next k's gap.
    """
    ks = list(ks or ([1] + list(C.K_RANGE)))
    rng = np.random.default_rng(random_state)

    def dispersion(X, k):
        if k == 1:
            centre = X.mean(axis=0, keepdims=True)
            return float(((X - centre) ** 2).sum())
        return float(KMeans(n_clusters=k, n_init=10,
                            random_state=random_state).fit(X).inertia_)

    # Reference box aligned to the principal axes of the observed data.
    from sklearn.decomposition import PCA
    p = PCA().fit(Z)
    Zr = Z @ p.components_.T
    lo, hi = Zr.min(axis=0), Zr.max(axis=0)

    obs_log, ref_log_mean, ref_log_sd = [], [], []
    for k in ks:
        obs_log.append(np.log(dispersion(Z, k)))
        refs = []
        for _ in range(n_refs):
            U = rng.uniform(lo, hi, size=Z.shape)
            refs.append(np.log(dispersion(U @ p.components_, k)))
        ref_log_mean.append(float(np.mean(refs)))
        ref_log_sd.append(float(np.std(refs)))

    gap = np.array(ref_log_mean) - np.array(obs_log)
    sk = np.array(ref_log_sd) * np.sqrt(1 + 1.0 / n_refs)

    chosen = ks[-1]
    for i in range(len(ks) - 1):
        if gap[i] >= gap[i + 1] - sk[i + 1]:
            chosen = ks[i]
            break

    return {
        "ks": ks,
        "gap": [round(float(g), 4) for g in gap],
        "s_k": [round(float(s), 4) for s in sk],
        "k_selected": int(chosen),
        "n_references": int(n_refs),
        "supports_no_structure": bool(chosen == 1),
        "interpretation": (
            "the gap statistic selects k = 1, i.e. the data is better described as one "
            "homogeneous group than as any partition tested"
            if chosen == 1 else
            "the gap statistic selects k = %d over the single-group null" % chosen),
    }


def knee_point(ks, sse):
    """Kneedle: the k whose SSE is furthest below the first-to-last chord.

    Turns "look for the elbow" into a reproducible number, which matters because
    on weakly clustered data different readers pick different elbows by eye.
    """
    x = np.asarray(ks, dtype=float)
    y = np.asarray(sse, dtype=float)
    xn = (x - x.min()) / (x.max() - x.min())
    yn = (y - y.min()) / (y.max() - y.min())
    # Perpendicular distance below the straight line joining the endpoints.
    dist = (yn[0] + (yn[-1] - yn[0]) * xn) - yn
    return int(x[int(np.argmax(dist))]), [round(float(d), 4) for d in dist]


# --------------------------------------------------------------------------
# k-means sweep
# --------------------------------------------------------------------------
def kmeans_sweep(Z, ks=C.K_RANGE, n_init=C.N_INIT, random_state=C.RANDOM_STATE):
    """Fit k-means for each k and collect SSE plus all three internal indices."""
    rows, models = [], {}
    for k in ks:
        km = KMeans(n_clusters=k, n_init=n_init, random_state=random_state).fit(Z)
        models[k] = km
        rows.append({"k": k, "sse": round(float(km.inertia_), 2),
                     **internal_indices(Z, km.labels_)})
    table = pd.DataFrame(rows).set_index("k")
    knee, dists = knee_point(list(table.index), table["sse"].to_numpy())
    report = {
        "table": table.reset_index().to_dict("records"),
        "sse_knee_k": knee,
        "knee_distances": dists,
        "best_silhouette_k": int(table["silhouette"].idxmax()),
        "best_davies_bouldin_k": int(table["davies_bouldin"].idxmin()),
        "best_calinski_k": int(table["calinski_harabasz"].idxmax()),
    }
    return models, table, report


# --------------------------------------------------------------------------
# Gaussian mixture (WEKA EM equivalent, including automatic k)
# --------------------------------------------------------------------------
def gmm_sweep(Z, ks=C.K_RANGE, random_state=C.RANDOM_STATE, cv_folds=C.CV_FOLDS):
    """Fit Gaussian mixtures and select k by BIC and by cross-validated log-likelihood.

    WEKA's EM picks k by ten-fold cross-validated log-likelihood; the same rule
    is reproduced here so the two toolchains can be compared directly, with BIC
    reported alongside as the standard penalised-likelihood criterion.
    """
    rows, models = [], {}
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    for k in ks:
        gm = GaussianMixture(n_components=k, covariance_type="full",
                             n_init=5, random_state=random_state).fit(Z)
        models[k] = gm
        folds = []
        for tr, te in kf.split(Z):
            g = GaussianMixture(n_components=k, covariance_type="full",
                                n_init=3, random_state=random_state).fit(Z[tr])
            folds.append(g.score(Z[te]))
        labels = gm.predict(Z)
        rows.append({
            "k": k,
            "bic": round(float(gm.bic(Z)), 1),
            "aic": round(float(gm.aic(Z)), 1),
            "cv_loglik": round(float(np.mean(folds)), 4),
            "cv_loglik_sd": round(float(np.std(folds)), 4),
            **internal_indices(Z, labels),
        })
    table = pd.DataFrame(rows).set_index("k")
    report = {
        "table": table.reset_index().to_dict("records"),
        "best_bic_k": int(table["bic"].idxmin()),
        "best_cv_loglik_k": int(table["cv_loglik"].idxmax()),
    }
    return models, table, report


# --------------------------------------------------------------------------
# Hierarchical
# --------------------------------------------------------------------------
def hierarchical(Z, ks=C.K_RANGE, methods=("ward", "average", "complete")):
    """Linkage matrices, cophenetic correlation, and cut-based labellings.

    Cophenetic correlation says how faithfully each linkage preserves the
    original distances; it is the standard way to justify choosing Ward rather
    than assuming it.
    """
    from scipy.spatial.distance import pdist

    D = pdist(Z, metric="euclidean")
    out, links, labels_by = {}, {}, {}
    for m in methods:
        Lk = linkage(Z, method=m) if m == "ward" else linkage(D, method=m)
        links[m] = Lk
        coph, _ = cophenet(Lk, D)
        rows = []
        labels_by[m] = {}
        for k in ks:
            lab = fcluster(Lk, t=k, criterion="maxclust") - 1
            labels_by[m][k] = lab
            sizes = np.bincount(lab, minlength=k)
            rows.append({"k": k,
                         "largest_cluster_pct": round(float(100 * sizes.max() / len(lab)), 1),
                         "n_singleton_clusters": int((sizes <= 2).sum()),
                         **internal_indices(Z, lab)})
        out[m] = {"cophenetic_r": round(float(coph), 4),
                  "table": rows,
                  "merge_heights_last10": [round(float(h), 3) for h in Lk[-10:, 2]]}

    best_coph = max(out, key=lambda m: out[m]["cophenetic_r"])

    # A linkage that puts almost everyone in one cluster is degenerate, however
    # well it preserves pairwise distances. Average linkage typically wins the
    # cophenetic comparison on continuum-like data precisely because it chains:
    # it hangs a few outliers off one enormous cluster, which reproduces the
    # distance matrix faithfully and segments nothing. So the reported cut uses
    # the best-correlating linkage that also produces a usable split.
    def degenerate(m):
        return any(r["largest_cluster_pct"] > 90 for r in out[m]["table"])

    usable = [m for m in out if not degenerate(m)]
    best_usable = (max(usable, key=lambda m: out[m]["cophenetic_r"])
                   if usable else best_coph)

    return links, labels_by, {
        "by_method": out,
        "best_cophenetic_method": best_coph,
        "degenerate_methods": [m for m in out if degenerate(m)],
        "reported_method": best_usable,
        "note": ("%s has the highest cophenetic correlation (%.3f) but is degenerate - it "
                 "leaves over 90%% of students in a single cluster at every k tested. The "
                 "reported hierarchical cut therefore uses %s."
                 % (best_coph, out[best_coph]["cophenetic_r"], best_usable)
                 if best_coph != best_usable else
                 "%s has the highest cophenetic correlation (%.3f) and produces a usable split."
                 % (best_coph, out[best_coph]["cophenetic_r"])),
    }


def agglomerative_labels(Z, k, linkage_method="ward"):
    """Direct sklearn agglomerative fit, kept for the final chosen k."""
    ac = AgglomerativeClustering(n_clusters=k, linkage=linkage_method)
    return ac.fit_predict(Z), ac


# --------------------------------------------------------------------------
# Mixed-data branch: Gower distance + k-medoids (PAM)
# --------------------------------------------------------------------------
def gower_matrix(df, numeric_cols, categorical_cols):
    """Gower dissimilarity for a mixed numeric / categorical table.

    Numeric contributions are absolute differences scaled by the column range;
    categorical contributions are 0/1 mismatches. Implemented directly because
    the `gower` package is not part of the Kaggle base image, and n is under a
    thousand so the full n-by-n matrix is cheap.
    """
    n = len(df)
    # float32 and in-place accumulation: the full matrix is O(n^2), so a cohort
    # of a few thousand students would otherwise allocate hundreds of MB in
    # temporaries alone. `buf` is reused for every column.
    total = np.zeros((n, n), dtype=np.float32)
    buf = np.empty((n, n), dtype=np.float32)
    weight = 0.0

    for c in numeric_cols:
        v = df[c].to_numpy(dtype=np.float32)
        rng = float(v.max() - v.min())
        if rng <= 0:
            continue
        np.subtract(v[:, None], v[None, :], out=buf)
        np.abs(buf, out=buf)
        buf /= rng
        total += buf
        weight += 1.0

    for c in categorical_cols:
        # Factorised codes compare as integers, which lets the mismatch indicator
        # be written straight into the float buffer with no string temporaries.
        codes = pd.factorize(df[c].astype(str))[0].astype(np.float32)
        np.subtract(codes[:, None], codes[None, :], out=buf)
        np.sign(buf, out=buf)
        np.abs(buf, out=buf)
        total += buf
        weight += 1.0

    if weight == 0:
        raise ValueError("Gower matrix needs at least one usable column.")
    total /= weight
    np.fill_diagonal(total, 0.0)
    return total


def kmedoids(D, k, random_state=C.RANDOM_STATE, max_iter=300, n_init=10):
    """Voronoi-iteration PAM over a precomputed distance matrix."""
    rng = np.random.default_rng(random_state)
    n = D.shape[0]
    best_labels, best_cost, best_medoids = None, np.inf, None

    for _ in range(n_init):
        # k-means++ style seeding on the distance matrix.
        medoids = [int(rng.integers(n))]
        while len(medoids) < k:
            d = D[:, medoids].min(axis=1) ** 2
            s = d.sum()
            medoids.append(int(rng.choice(n, p=d / s)) if s > 0 else int(rng.integers(n)))
        medoids = np.array(medoids)

        for _ in range(max_iter):
            labels = np.argmin(D[:, medoids], axis=1)
            new = medoids.copy()
            for j in range(k):
                members = np.flatnonzero(labels == j)
                if len(members):
                    new[j] = members[int(np.argmin(D[np.ix_(members, members)].sum(axis=1)))]
            if np.array_equal(new, medoids):
                break
            medoids = new

        labels = np.argmin(D[:, medoids], axis=1)
        cost = float(D[np.arange(n), medoids[labels]].sum())
        if cost < best_cost:
            best_labels, best_cost, best_medoids = labels, cost, medoids

    return best_labels, best_medoids, best_cost


def gower_sweep(D, ks=C.K_RANGE, random_state=C.RANDOM_STATE):
    """k-medoids over Gower distance for each k, scored with precomputed silhouette."""
    rows, labels_by = [], {}
    for k in ks:
        lab, med, cost = kmedoids(D, k, random_state=random_state)
        labels_by[k] = lab
        sil = (float(silhouette_score(D, lab, metric="precomputed"))
               if len(set(lab)) > 1 else None)
        rows.append({"k": k, "cost": round(cost, 2),
                     "silhouette_gower": None if sil is None else round(sil, 4),
                     "n_medoids": len(set(med))})
    table = pd.DataFrame(rows).set_index("k")
    return labels_by, table, {"table": table.reset_index().to_dict("records"),
                              "best_silhouette_k": int(table["silhouette_gower"].idxmax())}


# --------------------------------------------------------------------------
# Spectral cross-check
# --------------------------------------------------------------------------
def spectral_labels(Z, k, random_state=C.RANDOM_STATE):
    """Spectral clustering, which unlike k-means does not assume convex clusters."""
    sc = SpectralClustering(n_clusters=k, random_state=random_state,
                            affinity="nearest_neighbors", n_neighbors=15,
                            assign_labels="kmeans")
    return sc.fit_predict(Z)


# --------------------------------------------------------------------------
# Selection
# --------------------------------------------------------------------------
def select_k(km_report, gmm_report, hier_report, gap_report=None, km_table=None):
    """Combine the independent k signals into one decision with an audit trail.

    Each signal casts one vote and the modal vote wins. Ties are broken by
    silhouette, which the project proposal names as the primary internal
    criterion, and only then by parsimony. The full vote record is returned
    because on data this weakly structured the *disagreement* between criteria
    is a finding in its own right, not something to resolve silently.

    The gap statistic is reported separately rather than as a vote: it answers a
    different question (is there any structure at all) and including k = 1 in a
    vote about how many clusters to profile would confuse the two.
    """
    votes = {
        "kmeans_sse_elbow": km_report["sse_knee_k"],
        "kmeans_silhouette": km_report["best_silhouette_k"],
        "kmeans_davies_bouldin": km_report["best_davies_bouldin_k"],
        "kmeans_calinski": km_report["best_calinski_k"],
        "gmm_bic": gmm_report["best_bic_k"],
        "gmm_cv_loglik": gmm_report["best_cv_loglik_k"],
    }
    # Vote with the linkage the hierarchical step reports, not a hardcoded one,
    # so a degenerate linkage can never cast a vote.
    hm = hier_report.get("reported_method", "ward")
    rows = hier_report["by_method"][hm]["table"]
    best_row = max(rows, key=lambda r: (r["silhouette"] if r["silhouette"] is not None else -1))
    votes["hierarchical_%s_silhouette" % hm] = int(best_row["k"])

    tally = {}
    for v in votes.values():
        tally[v] = tally.get(v, 0) + 1
    top = max(tally.values())
    tied = sorted(k for k, c in tally.items() if c == top)

    if len(tied) > 1 and km_table is not None:
        chosen = max(tied, key=lambda k: (float(km_table.loc[k, "silhouette"]), -k))
        tie_rule = "tie between %s broken by silhouette" % tied
    else:
        chosen = tied[0]
        tie_rule = "clear modal vote" if len(tied) == 1 else "tie broken by parsimony"

    out = {"votes": votes, "tally": tally, "k_selected": int(chosen),
           "n_signals": len(votes), "n_agreeing": int(top),
           "tied_candidates": tied, "tie_break": tie_rule,
           "criteria_disagree": bool(len(set(votes.values())) > 2)}
    if gap_report is not None:
        out["gap_statistic"] = gap_report
    return out


def structure_verdict(silhouette):
    """Plain-language reading of how real the cluster structure is.

    Thresholds follow Kaufman and Rousseeuw's conventional bands. Stating them
    up front prevents a weak solution from being written up as a strong one.
    """
    if silhouette is None:
        return "no valid silhouette"
    if silhouette >= 0.50:
        return "reasonable structure"
    if silhouette >= 0.25:
        return "weak structure that could be artificial"
    return ("no substantial structure: the partition is a convenience segmentation of a "
            "continuum, not evidence of naturally separated groups")

### `ml/validate.py` - Cluster validation (methodology 5.6): stability, consensus and external checks.

In [ ]:
%%writefile ml/validate.py
# -*- coding: utf-8 -*-
"""
Cluster validation (methodology 5.6): stability, consensus and external checks.

Internal indices alone cannot tell you whether a partition is real. Three
independent lines of evidence are added here:

  * bootstrap stability - re-cluster resamples and measure agreement with the
    reference labelling by adjusted Rand index. Unstable clusters are an
    artifact of one sample draw.
  * consensus - how often each pair of students lands together across
    resamples, summarised per cluster as a co-assignment rate.
  * classes-to-clusters - agreement with each held-out background variable
    (year, gender, living arrangement, CGPA band). None of these entered the
    feature set, so above-chance agreement is genuine external validation and
    at-chance agreement is a real, reportable negative.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import (adjusted_mutual_info_score, adjusted_rand_score,
                             normalized_mutual_info_score, silhouette_samples)

from . import config as C


# --------------------------------------------------------------------------
# Stability
# --------------------------------------------------------------------------
def bootstrap_stability(Z, k, reference_labels, b=C.BOOTSTRAP_B,
                        random_state=C.RANDOM_STATE, sample_frac=0.80):
    """Cluster B subsamples; compare each to the reference on shared rows (ARI).

    Subsampling rather than resampling with replacement keeps duplicate points
    from inflating agreement artificially.
    """
    rng = np.random.default_rng(random_state)
    n = Z.shape[0]
    size = int(round(sample_frac * n))
    aris, amis = [], []

    for i in range(b):
        idx = rng.choice(n, size=size, replace=False)
        km = KMeans(n_clusters=k, n_init=10, random_state=int(rng.integers(1 << 31))).fit(Z[idx])
        aris.append(adjusted_rand_score(reference_labels[idx], km.labels_))
        amis.append(adjusted_mutual_info_score(reference_labels[idx], km.labels_))

    aris = np.asarray(aris)
    return {
        "n_resamples": int(b),
        "sample_fraction": sample_frac,
        "ari_mean": round(float(aris.mean()), 4),
        "ari_sd": round(float(aris.std()), 4),
        "ari_p05": round(float(np.percentile(aris, 5)), 4),
        "ari_median": round(float(np.median(aris)), 4),
        "ari_p95": round(float(np.percentile(aris, 95)), 4),
        "ami_mean": round(float(np.mean(amis)), 4),
        "pct_ari_above_0.5": round(float(100 * (aris > 0.5).mean()), 1),
        "pct_ari_above_0.75": round(float(100 * (aris > 0.75).mean()), 1),
    }


def seed_stability(Z, k, trials=C.SEED_TRIALS):
    """Re-run k-means from different seeds; report pairwise ARI across runs.

    Answers a narrower question than the bootstrap: given this exact sample, is
    the solution an artifact of initialisation?
    """
    labs = []
    for s in range(trials):
        labs.append(KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(Z))
    pair = [adjusted_rand_score(labs[i], labs[j])
            for i in range(trials) for j in range(i + 1, trials)]
    pair = np.asarray(pair)
    return {
        "n_seeds": int(trials),
        "pairwise_ari_mean": round(float(pair.mean()), 4),
        "pairwise_ari_min": round(float(pair.min()), 4),
        "pct_identical": round(float(100 * (pair > 0.99).mean()), 1),
    }


def consensus(Z, k, b=100, random_state=C.RANDOM_STATE, sample_frac=0.80):
    """Pairwise co-assignment rate across resamples.

    Returns the consensus matrix plus, for the reference labelling, the mean
    within-cluster co-assignment - a per-cluster reliability figure that says
    which of the profiles is trustworthy and which is a residual bucket.
    """
    rng = np.random.default_rng(random_state)
    n = Z.shape[0]
    size = int(round(sample_frac * n))
    together = np.zeros((n, n), dtype=np.float32)
    seen = np.zeros((n, n), dtype=np.float32)

    for _ in range(b):
        idx = rng.choice(n, size=size, replace=False)
        lab = KMeans(n_clusters=k, n_init=10,
                     random_state=int(rng.integers(1 << 31))).fit_predict(Z[idx])
        same = (lab[:, None] == lab[None, :]).astype(np.float32)
        together[np.ix_(idx, idx)] += same
        seen[np.ix_(idx, idx)] += 1.0

    with np.errstate(invalid="ignore", divide="ignore"):
        M = np.where(seen > 0, together / seen, np.nan)
    np.fill_diagonal(M, 1.0)
    return M


def consensus_by_cluster(M, labels):
    """Mean co-assignment within each reference cluster."""
    out = {}
    for c in sorted(set(labels)):
        idx = np.flatnonzero(labels == c)
        if len(idx) < 2:
            out[int(c)] = None
            continue
        block = M[np.ix_(idx, idx)]
        iu = np.triu_indices(len(idx), k=1)
        out[int(c)] = round(float(np.nanmean(block[iu])), 4)
    return out


# --------------------------------------------------------------------------
# External / classes-to-clusters
# --------------------------------------------------------------------------
def cramers_v(table):
    """Bias-corrected Cramer's V for a contingency table."""
    chi2 = stats.chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    r, k = table.shape
    phi2 = chi2 / n
    phi2c = max(0.0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    kc = k - (k - 1) ** 2 / (n - 1)
    denom = min(kc - 1, rc - 1)
    return float(np.sqrt(phi2c / denom)) if denom > 0 else float("nan")


def classes_to_clusters(labels, background, variables=("year", "gender", "living", "cgpa", "department")):
    """Test each held-out background variable against the cluster labelling.

    Reports chi-square, Cramer's V, ARI and NMI. The classes-to-clusters error
    (WEKA's own measure) is included: assign each cluster to its majority class
    and count how many students that misclassifies.
    """
    out = {}
    for var in variables:
        if var not in background.columns:
            continue
        y = background[var].astype(str)
        ct = pd.crosstab(pd.Series(labels, index=y.index, name="cluster"), y)
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        majority = ct.idxmax(axis=1)
        correct = int(sum(ct.loc[c, majority[c]] for c in ct.index))
        codes = pd.Categorical(y).codes
        out[var] = {
            "chi2": round(float(chi2), 2),
            "dof": int(dof),
            "p": float(p),
            "cramers_v": round(cramers_v(ct), 4),
            "adjusted_rand": round(float(adjusted_rand_score(codes, labels)), 4),
            "nmi": round(float(normalized_mutual_info_score(codes, labels)), 4),
            "classes_to_clusters_accuracy": round(correct / len(y), 4),
            "majority_class_baseline": round(float(y.value_counts(normalize=True).max()), 4),
            "cluster_majority_class": {str(c): str(majority[c]) for c in ct.index},
        }
    return out


def silhouette_breakdown(Z, labels):
    """Per-cluster silhouette, plus the share of negatively scored members.

    A cluster whose members mostly score below zero is closer to another cluster
    than to its own and should not be presented as a coherent group.
    """
    s = silhouette_samples(Z, labels)
    out = {}
    for c in sorted(set(labels)):
        m = labels == c
        out[int(c)] = {
            "n": int(m.sum()),
            "mean_silhouette": round(float(s[m].mean()), 4),
            "pct_negative": round(float(100 * (s[m] < 0).mean()), 1),
        }
    return {"overall_mean": round(float(s.mean()), 4),
            "pct_negative_overall": round(float(100 * (s < 0).mean()), 1),
            "by_cluster": out}, s


def compare_algorithms(labelings):
    """Pairwise ARI between the labellings produced by different algorithms.

    High agreement means the partition is a property of the data; low agreement
    means it is a property of the algorithm, which is itself the finding.
    """
    names = list(labelings)
    M = pd.DataFrame(index=names, columns=names, dtype=float)
    for i, a in enumerate(names):
        for b in names:
            M.loc[a, b] = round(float(adjusted_rand_score(labelings[a], labelings[b])), 4)
    return M

### `ml/lexicon.py` - The frozen code-mixed stressor lexicon (methodology 5.3).

In [ ]:
%%writefile ml/lexicon.py
# -*- coding: utf-8 -*-
"""
The frozen code-mixed stressor lexicon (methodology 5.3).

Thirteen themes, each a regular-expression pattern list covering English terms,
Bangla script and romanised Bangla. Multi-label by design: one answer can carry
several themes, because "lab reports and money problems" is two stressors, not
a tie to be broken.

Why a lexicon rather than an off-the-shelf NLP pipeline, restated here so the
choice travels with the code:
  * the answers are short (median 5 words, ~28% are two words or fewer), so
    contextual embeddings have almost no context to work with;
  * roughly one answer in eight carries Bangla script, and a further ~2% is
    romanised Bangla, which English stemmers and stopword lists silently damage;
  * the question asks for a *source of stress*, so sentiment is negative by
    construction and carries no between-student variance.

This module is frozen: the patterns are the published instrument. Changing them
after seeing results would invalidate the prevalence figures, so any edit should
be a new version with the whole pipeline re-run.
"""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

LEXICON_VERSION = "1.0"

THEMES = {
    "Lab & coursework load": [
        r"\blabs?\b", r"lab ?report", r"lab ?task", r"lab ?test", r"labwork", r"lap ?report",
        r"\bassignment", r"\bprojects?\b", r"presentation", r"sessional", r"\bquiz", r"\bviva",
        r"\bcts?\b", r"workload", r"\bdrawing\b", r"\bworkshop\b", r"\bstudio\b",
        r"ল্যাব", r"রিপোর্ট", r"অ্যাসাইনমেন্ট"],
    "Exams & results": [
        r"\bexam", r"\bresults?\b", r"\bcgpa\b", r"\bcg\b", r"\bgpa\b", r"\bmid ?term",
        r"term final", r"\bmarks?\b", r"\bgrade", r"\bbacklog\b", r"\bfail",
        r"পরীক্ষা", r"রেজাল্ট", r"ফলাফল", r"সিজি"],
    "Time & schedule pressure": [
        r"time manage", r"\bschedule", r"\broutine\b", r"8 ?am", r"short time", r"little time",
        r"session ?jam", r"\bsyllabus\b", r"deadline", r"\bpile", r"\battendance\b",
        r"\bshort pl\b", r"\bbreak\b", r"সময়", r"রুটিন", r"ক্লাস", r"সিলেবাস"],
    "Career & job uncertainty": [
        r"\bjobs?\b", r"\bcareer\b", r"\bfuture\b", r"graduat", r"\bskills?\b", r"placement",
        r"higher stud", r"\bscholarship\b", r"\babroad\b", r"job market",
        r"চাকরি", r"ক্যারিয়ার", r"ভবিষ্যৎ", r"স্কিল"],
    "Financial stress": [
        r"financ", r"\bmoney\b", r"\btaka\b", r"\bcosts?\b", r"\bexpens", r"tuition",
        r"middle class", r"\bincome\b", r"আর্থিক", r"টাকা", r"খরচ"],
    "Family & homesickness": [
        r"\bfamily\b", r"\bhome\b", r"homesick", r"away from home", r"far from home",
        r"\bparents?\b", r"\balone\b", r"lonel", r"\bmiss(ing)?\b",
        r"পরিবার", r"বাসা", r"একা", r"একাকিত্ব", r"দূরে"],
    "Teachers & teaching quality": [
        r"\bteachers?\b", r"\bfaculty\b", r"\bsir\b", r"instructor", r"lectur",
        r"can'?t understand", r"not understand", r"\bguideline", r"শিক্ষক", r"স্যার"],
    "Peers, comparison & campus climate": [
        r"compar", r"keep up with", r"\bpeers?\b", r"classmate", r"selfish", r"\btoxic\b",
        r"ragging", r"\bfriends?\b", r"politic", r"\bbully", r"তুলনা", r"বন্ধু", r"রাজনীতি"],
    "Living conditions & food": [
        r"\bfoods?\b", r"\bmess\b", r"\bhall\b", r"canteen", r"\bmeals?\b", r"breakfast",
        r"hostel", r"\bwater\b", r"transport", r"commut", r"\blifestyle\b",
        r"খাবার", r"হল", r"মেস", r"নাস্তা"],
    "Sleep & health": [
        r"\bsleep", r"insomnia", r"\bhealth", r"\bsick\b", r"\btired\b", r"exhaust",
        r"depress", r"anxiet", r"\bmental\b", r"ঘুম", r"স্বাস্থ্য"],
    "Self-regulation & procrastination": [
        r"procrastinat", r"irregular", r"\blazy\b", r"laziness", r"motivat", r"\bfocus\b",
        r"distract", r"\bdiscipline\b", r"অলস", r"মনোযোগ"],
    "Curriculum & non-departmental courses": [
        r"non.?dept", r"non.?departmental", r"memoriz", r"rote", r"curriculum",
        r"course ?load", r"credit hour", r"\bhum\b"],
    "Generic academic pressure": [
        r"academic", r"study pressure", r"\bstudies\b", r"porasuna", r"porashona",
        r"পড়াশোনা", r"পড়ালেখা", r"পড়া", r"চাপ"],
}

THEME_NAMES = list(THEMES)

#: Compiled once; the tagger runs over ~1,700 answers twice per pipeline run.
_COMPILED = {t: [re.compile(p, re.IGNORECASE) for p in pats] for t, pats in THEMES.items()}

BANGLA_RE = re.compile(r"[ঀ-৿]")
LATIN_RE = re.compile(r"[A-Za-z]")

#: Romanised-Bangla cues, used only to size that slice of the corpus.
ROMANISED_CUES = [
    "porasuna", "porashona", "porar", "chap", "tension", "ovab", "obhab", "somoy",
    "onek", "kichu", "amar", "ami", "kore", "kora", "hoy", "nai", "valo", "bhalo",
    "khub", "beshi", "besi", "problem er", "tar por",
]


def tag_text(text):
    """Return the list of themes matched in one answer."""
    s = str(text)
    return [t for t, pats in _COMPILED.items() if any(p.search(s) for p in pats)]


def answered_mask(series):
    """True where the free-text field has any non-whitespace content."""
    return series.fillna("").astype(str).str.strip().str.len() > 0


def tag_frame(series):
    """Multi-label 0/1 theme matrix for a free-text column (unanswered rows are all-zero)."""
    mask = answered_mask(series)
    txt = series.fillna("").astype(str)
    tags = [tag_text(t) if a else [] for t, a in zip(txt, mask)]
    T = pd.DataFrame({th: [int(th in ts) for ts in tags] for th in THEME_NAMES},
                     index=series.index)
    return T, mask, tags


def language_of(text):
    bn = bool(BANGLA_RE.search(text))
    lat = bool(LATIN_RE.search(text))
    if bn and lat:
        return "mixed"
    if bn:
        return "bangla"
    return "latin"


def text_profile(series, mask):
    """Length and script composition of one free-text field."""
    s = series.fillna("").astype(str)[mask]
    wl = s.str.split().str.len()
    lc = s.map(language_of).value_counts()
    # Romanised Bangla is counted only among Latin-script answers: an answer that
    # already carries Bangla script is captured by the script counts above.
    low = s.str.lower()
    latin_only = s.map(language_of) == "latin"
    romanised = int(low[latin_only].apply(
        lambda t: any(c in t for c in ROMANISED_CUES)).sum())
    n = len(s)
    return {
        "n_answered": int(mask.sum()),
        "response_rate_pct": round(float(100 * mask.mean()), 1),
        "words_mean": round(float(wl.mean()), 1),
        "words_median": float(wl.median()),
        "words_max": int(wl.max()),
        "chars_mean": round(float(s.str.len().mean()), 1),
        "pct_le2_words": round(float(100 * (wl <= 2).mean()), 1),
        "pct_ge10_words": round(float(100 * (wl >= 10).mean()), 1),
        "lang_latin_pct": round(float(100 * lc.get("latin", 0) / n), 1),
        "lang_bangla_pct": round(float(100 * lc.get("bangla", 0) / n), 1),
        "lang_mixed_pct": round(float(100 * lc.get("mixed", 0) / n), 1),
        "romanised_n": romanised,
        "romanised_pct": round(float(100 * romanised / n), 1),
        "distinct_answers": int(s.nunique()),
    }


def coverage(T, mask):
    """Share of answered rows receiving at least one theme, and themes per answer."""
    counts = T.to_numpy().sum(axis=1)[mask.to_numpy()]
    return {
        "coverage_pct": round(float(100 * (counts >= 1).mean()), 1),
        "mean_themes_per_answer": round(float(counts.mean()), 2),
        "pct_multi_theme": round(float(100 * (counts >= 2).mean()), 1),
        "n_uncovered": int((counts == 0).sum()),
    }


def prevalence(T, mask):
    """Percentage of answered students mentioning each theme, descending."""
    sub = T[mask.to_numpy()]
    return (100 * sub.mean()).round(1).sort_values(ascending=False)


def export_patterns():
    """The lexicon as a flat table, so the frozen instrument ships with the report."""
    rows = []
    for t, pats in THEMES.items():
        rows.append({"theme": t, "n_patterns": len(pats), "patterns": " | ".join(pats)})
    return pd.DataFrame(rows).set_index("theme")

### `ml/textmodel.py` - Unsupervised cross-checks over the free text, plus the two pre-specified null tests.

In [ ]:
%%writefile ml/textmodel.py
# -*- coding: utf-8 -*-
"""
Unsupervised cross-checks over the free text, plus the two pre-specified null tests.

The lexicon in `lexicon.py` is the primary text method. This module supplies the
corroboration the methodology promises - a TF-IDF/NMF topic model and an LDA run
- and the two checks that were specified in advance so a null result cannot be
quietly dropped:

  1. non-response signal: do students who skip the free-text field differ in
     measured strain from those who answer?
  2. answer-length signal: does how much a student writes correlate with strain?

Both are reported whatever they show.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from . import config as C
from . import lexicon as LX


def tfidf_matrix(texts, max_features=1500, min_df=3):
    """TF-IDF over the answered rows.

    Deliberately Latin-script only, with a token pattern requiring three or more
    letters. That is a documented limitation, not an oversight: this branch is
    the cross-check, and the lexicon is what covers the Bangla-script answers.
    """
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), min_df=min_df,
                          stop_words="english", token_pattern=r"[a-zA-Z][a-zA-Z']{2,}")
    X = vec.fit_transform(texts)
    return X, vec


def nmf_topics(X, vec, n_components=6, random_state=C.RANDOM_STATE):
    """Non-negative matrix factorisation over TF-IDF; returns topic terms and shares."""
    nmf = NMF(n_components=n_components, random_state=random_state,
              init="nndsvda", max_iter=800)
    W = nmf.fit_transform(X)
    terms = np.array(vec.get_feature_names_out())
    dominant = W.argmax(axis=1)
    topics = []
    for i, comp in enumerate(nmf.components_):
        topics.append({
            "topic": i,
            "top_terms": terms[np.argsort(comp)[::-1][:10]].tolist(),
            "share_pct": round(float(100 * (dominant == i).mean()), 1),
        })
    return topics, W, nmf


def lda_topics(texts, n_components=6, random_state=C.RANDOM_STATE, min_df=3):
    """LDA over raw counts, as a second opinion on the NMF solution."""
    cv = CountVectorizer(max_features=1500, min_df=min_df, stop_words="english",
                         token_pattern=r"[a-zA-Z][a-zA-Z']{2,}")
    Xc = cv.fit_transform(texts)
    lda = LatentDirichletAllocation(n_components=n_components, random_state=random_state,
                                    learning_method="batch", max_iter=50)
    Wl = lda.fit_transform(Xc)
    terms = np.array(cv.get_feature_names_out())
    dom = Wl.argmax(axis=1)
    topics = []
    for i, comp in enumerate(lda.components_):
        topics.append({
            "topic": i,
            "top_terms": terms[np.argsort(comp)[::-1][:10]].tolist(),
            "share_pct": round(float(100 * (dom == i).mean()), 1),
        })
    return {"topics": topics, "perplexity": round(float(lda.perplexity(Xc)), 1),
            "vocabulary": int(Xc.shape[1])}


def topic_model_report(texts):
    """Full topic-model cross-check, including how much text it cannot represent."""
    X, vec = tfidf_matrix(texts)
    zero_rows = int(np.asarray((X.sum(axis=1) == 0)).ravel().sum())
    topics, W, nmf = nmf_topics(X, vec)
    return {
        "tfidf_shape": list(X.shape),
        "vocabulary": int(X.shape[1]),
        "rows_with_no_features": zero_rows,
        "rows_with_no_features_pct": round(100 * zero_rows / X.shape[0], 1),
        "nmf_topics": topics,
        "lda": lda_topics(texts),
        "note": ("Rows with no features are answers written entirely in Bangla script or in "
                 "tokens below the min_df threshold. They are covered by the lexicon but "
                 "invisible to this Latin-script cross-check."),
    }, (X, vec, nmf, W)


def top_terms(texts, n=25, min_df=2):
    """Raw document-frequency ranking, for the descriptive table."""
    cv = CountVectorizer(min_df=min_df, stop_words="english",
                         token_pattern=r"[a-zA-Z][a-zA-Z']{2,}", binary=True)
    Xc = cv.fit_transform(texts)
    freq = np.asarray(Xc.sum(axis=0)).ravel()
    terms = np.array(cv.get_feature_names_out())
    order = np.argsort(freq)[::-1][:n]
    return [{"term": str(terms[i]), "n_answers": int(freq[i])} for i in order]


# --------------------------------------------------------------------------
# Pre-specified null checks
# --------------------------------------------------------------------------
def null_checks(strain, texts, mask):
    """The two checks specified in advance in methodology 5.3.

    `strain` is any per-student strain summary (the composite mean is used, as a
    descriptive index only). Both tests are reported with effect sizes, because
    at n ~ 1000 a p-value alone will call a trivial difference significant.
    """
    answered = strain[mask.to_numpy()]
    skipped = strain[~mask.to_numpy()]

    t, p = stats.ttest_ind(answered, skipped, equal_var=False)
    pooled = np.sqrt((answered.var(ddof=1) + skipped.var(ddof=1)) / 2)
    d = float((answered.mean() - skipped.mean()) / pooled) if pooled > 0 else float("nan")

    wl = texts.fillna("").astype(str).str.split().str.len()[mask.to_numpy()]
    rho, p_rho = stats.spearmanr(wl, answered)

    return {
        "non_response": {
            "n_answered": int(mask.sum()),
            "n_skipped": int((~mask).sum()),
            "mean_strain_answered": round(float(answered.mean()), 3),
            "mean_strain_skipped": round(float(skipped.mean()), 3),
            "difference": round(float(answered.mean() - skipped.mean()), 3),
            "welch_t": round(float(t), 3),
            "p": float(p),
            "cohens_d": round(d, 3),
            "verdict": ("no usable signal: skipping the free-text field does not indicate "
                        "different measured strain"
                        if p >= 0.05 or abs(d) < 0.2 else
                        "students who skip the field differ in measured strain"),
        },
        "answer_length": {
            "spearman_rho": round(float(rho), 3),
            "p": float(p_rho),
            "verdict": ("no usable signal: answer length does not track measured strain"
                        if p_rho >= 0.05 or abs(rho) < 0.1 else
                        "answer length tracks measured strain"),
        },
    }


def theme_strain_association(T, mask, strain, min_n=15):
    """Per-theme difference in strain between mentioners and non-mentioners.

    Bonferroni-corrected across the themes actually tested, because thirteen
    uncorrected t-tests will manufacture a significant result on their own.
    """
    rows = []
    m_all = mask.to_numpy()
    for t in LX.THEME_NAMES:
        flag = T[t].to_numpy().astype(bool)
        with_t = strain[flag & m_all]
        without = strain[(~flag) & m_all]
        if len(with_t) < min_n:
            continue
        stat, p = stats.ttest_ind(with_t, without, equal_var=False)
        pooled = np.sqrt((with_t.var(ddof=1) + without.var(ddof=1)) / 2)
        rows.append({
            "theme": t,
            "n_mentioning": int(len(with_t)),
            "mean_with": round(float(with_t.mean()), 3),
            "mean_without": round(float(without.mean()), 3),
            "difference": round(float(with_t.mean() - without.mean()), 3),
            "cohens_d": round(float((with_t.mean() - without.mean()) / pooled), 3) if pooled else None,
            "p": float(p),
        })
    if not rows:
        return {"bonferroni_alpha": None, "rows": []}
    alpha = 0.05 / len(rows)
    for r in rows:
        r["significant_bonferroni"] = bool(r["p"] < alpha)
    rows.sort(key=lambda r: -abs(r["difference"]))
    return {"n_tests": len(rows), "bonferroni_alpha": round(alpha, 5), "rows": rows}


def theme_by_group(T, mask, groups, min_n=20):
    """Theme prevalence within each level of a background variable."""
    out = {}
    m = mask.to_numpy()
    for g in pd.unique(groups):
        sel = m & (groups == g).to_numpy()
        if sel.sum() < min_n:
            continue
        out[str(g)] = {"n": int(sel.sum()),
                       **{t: round(float(100 * T.loc[sel, t].mean()), 1) for t in LX.THEME_NAMES}}
    return out

### `ml/profile.py` - Cluster profiling and plain-language naming (methodology 5.7).

In [ ]:
%%writefile ml/profile.py
# -*- coding: utf-8 -*-
"""
Cluster profiling and plain-language naming (methodology 5.7).

Names are derived from the numbers, not chosen first. `name_clusters` ranks each
cluster's item z-scores against the grand mean and builds the label from the
items that actually separate it, so a reader can check the label against the
profile table. Ordering is by overall strain so P1 is always the lowest-strain
profile regardless of the arbitrary integer k-means assigns.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

from . import config as C
from . import lexicon as LX


#: Short descriptors used when an item is a cluster's defining high/low feature.
_HIGH_TAG = {
    "MissMeal": "skipped meals",
    "PileUp": "coursework backlog",
    "SleepLoss": "sleep sacrifice",
    "LabStress": "lab strain",
    "ExamWorry": "exam anxiety",
    "ResultDemotiv": "result-driven demotivation",
    "CGPACompare": "peer CGPA comparison",
    "AskTeacher": "low help-seeking",
    "Feedback": "unsupportive feedback",
    "Financial": "financial pressure",
    "JobWorry": "career anxiety",
    "SocioPol": "socio-political worry",
}


def profile_table(items, labels):
    """Mean of every item within each cluster, plus the grand mean column."""
    df = items.copy()
    df["cluster"] = labels
    prof = df.groupby("cluster")[C.ITEMS].mean().round(3)
    prof.loc["ALL"] = items[C.ITEMS].mean().round(3)
    return prof


def zprofile(items, labels):
    """Cluster means expressed as z-scores of the item distribution.

    This is what the naming rule reads. Raw means are not comparable across
    items, because item means range from about 2.4 to 4.4 on the aligned scale.
    """
    mu = items[C.ITEMS].mean()
    sd = items[C.ITEMS].std()
    df = items.copy()
    df["cluster"] = labels
    prof = df.groupby("cluster")[C.ITEMS].mean()
    return ((prof - mu) / sd).round(3)


def order_by_strain(items, labels):
    """Map raw cluster ids to strain-ordered positions (0 = lowest strain)."""
    strain = pd.Series(items[C.ITEMS].mean(axis=1).to_numpy()).groupby(labels).mean()
    return {int(c): i for i, c in enumerate(strain.sort_values().index)}


def name_clusters(items, labels, z_threshold=0.30):
    """Derive a label per cluster from its most distinctive items.

    Rule: take the item z-scores of each cluster, keep those beyond
    +/- z_threshold, and build the name from the two strongest highs and the
    strongest low. A cluster with nothing beyond the threshold is named as the
    middle group, which is an honest outcome on weakly separated data.
    """
    Zp = zprofile(items, labels)
    order = order_by_strain(items, labels)
    k = len(order)
    names, detail = {}, {}

    for c in Zp.index:
        row = Zp.loc[c].sort_values(ascending=False)
        highs = [i for i in row.index if row[i] >= z_threshold]
        lows = [i for i in row.index if row[i] <= -z_threshold]
        pos = order[int(c)]
        rank = "P%d" % (pos + 1)

        # `row` is sorted descending, so lows[-1] is the most strongly suppressed item.
        if not highs and not lows:
            label = "%s - Middle / undifferentiated" % rank
        elif not highs:
            label = "%s - Lower strain, esp. low %s" % (rank, _HIGH_TAG[lows[-1]])
        else:
            bits = ["high " + _HIGH_TAG[h] for h in highs[:2]]
            if lows:
                bits.append("low " + _HIGH_TAG[lows[-1]])
            label = "%s - %s" % (rank, ", ".join(bits))

        names[int(c)] = label
        detail[int(c)] = {
            "strain_rank": pos + 1,
            "of": k,
            "elevated_items": [{"item": i, "z": float(row[i]), "label": C.ITEM_LABEL[i]}
                               for i in highs],
            "suppressed_items": [{"item": i, "z": float(row[i]), "label": C.ITEM_LABEL[i]}
                                 for i in lows],
        }
    return names, detail, Zp


def composition(labels, background, variables=("year", "cgpa", "gender", "living", "department"),
                orders=None):
    """Row-percentage composition of each cluster across background variables."""
    orders = orders or {"year": C.YEAR_ORD, "cgpa": C.CGPA_ORD,
                        "gender": C.GENDER_ORD, "living": C.LIVE_ORD}
    out = {}
    s = pd.Series(labels, index=background.index, name="cluster")
    for var in variables:
        if var not in background.columns:
            continue
        ct = pd.crosstab(s, background[var].astype(str), normalize="index") * 100
        order = orders.get(var)
        if order:
            keep = [c for c in order if c in ct.columns]
            ct = ct[keep + [c for c in ct.columns if c not in keep]]
        raw = pd.crosstab(s, background[var].astype(str))
        chi2, p, dof, _ = stats.chi2_contingency(raw)
        out[var] = {
            "pct": {int(c): {str(k): round(float(ct.loc[c, k]), 1) for k in ct.columns}
                    for c in ct.index},
            "chi2": round(float(chi2), 2), "dof": int(dof), "p": float(p),
        }
    return out


def theme_by_cluster(T, mask, labels):
    """Theme prevalence within each cluster - the triangulation step.

    Text themes were never clustering features, so agreement between what a
    cluster scores high on and what its members volunteer is independent
    corroboration rather than circular reasoning.
    """
    out = {}
    m = mask.to_numpy()
    for c in sorted(set(labels)):
        sel = m & (labels == c)
        out[int(c)] = {"n_answered": int(sel.sum()),
                       **{t: round(float(100 * T.loc[sel, t].mean()), 1) for t in LX.THEME_NAMES}}
    return out


def cluster_summary(items, labels, background, names):
    """One row per cluster: size, strain, modal background categories."""
    strain = items[C.ITEMS].mean(axis=1)
    rows = []
    for c in sorted(set(labels)):
        m = labels == c
        rows.append({
            "cluster": int(c),
            "profile": names[int(c)],
            "n": int(m.sum()),
            "pct_of_sample": round(float(100 * m.mean()), 1),
            "mean_strain": round(float(strain[m].mean()), 3),
            "sd_strain": round(float(strain[m].std()), 3),
            "modal_year": background.loc[m, "year"].mode().iat[0],
            "modal_cgpa": background.loc[m, "cgpa"].mode().iat[0],
            "modal_living": background.loc[m, "living"].mode().iat[0],
            "pct_female": round(float(100 * (background.loc[m, "gender"] == "Female").mean()), 1),
            "pct_backlog": round(float(100 * background.loc[m, "backlog"].mean()), 1),
        })
    return pd.DataFrame(rows).set_index("cluster")


def discriminating_items(items, labels, top=5):
    """Rank items by how much of their variance the partition explains (eta-squared).

    Answers the proposal's second question - which features most strongly
    differentiate the clusters - with an effect size rather than an F-statistic,
    so the answer does not depend on n.
    """
    rows = []
    for it in C.ITEMS:
        groups = [items.loc[labels == c, it].to_numpy() for c in sorted(set(labels))]
        f, p = stats.f_oneway(*groups)
        grand = items[it].mean()
        ss_between = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
        ss_total = ((items[it] - grand) ** 2).sum()
        rows.append({
            "item": it, "label": C.ITEM_LABEL[it],
            "f": round(float(f), 2), "p": float(p),
            "eta_squared": round(float(ss_between / ss_total), 4),
        })
    df = pd.DataFrame(rows).set_index("item").sort_values("eta_squared", ascending=False)
    return df, df.index[:top].tolist()


def recommendations(names, detail, theme_pct):
    """Map each profile to the support action its defining items point at.

    Kept as an explicit item-to-action table so the recommendation is traceable
    to a measurement rather than to the author's intuition.
    """
    action = {
        "PileUp": "coursework scheduling / deadline spreading across courses",
        "SleepLoss": "workload audit on lab-report turnaround times",
        "LabStress": "lab preparation support and clearer marking rubrics",
        "ExamWorry": "exam-anxiety workshops and low-stakes practice assessment",
        "ResultDemotiv": "post-result advising contact, not just grade release",
        "CGPACompare": "de-emphasise public ranking; individual progress feedback",
        "AskTeacher": "structured, low-barrier office hours and named advisers",
        "Feedback": "feedback-quality training and turnaround-time targets",
        "Financial": "signpost hardship funds, stipends and paid on-campus work",
        "JobWorry": "early careers advising, internships, alumni contact",
        "SocioPol": "acknowledge context; general counselling availability",
        "MissMeal": "timetable review around meal windows; canteen hours",
    }
    out = {}
    for c, d in detail.items():
        acts = [action[e["item"]] for e in d["elevated_items"][:3] if e["item"] in action]
        acts += [action[e["item"]] for e in d["suppressed_items"][:1]
                 if e["item"] in action and e["item"] in ("AskTeacher", "Feedback")]
        top_themes = sorted(
            [(t, v) for t, v in theme_pct.get(int(c), {}).items() if t in LX.THEME_NAMES],
            key=lambda kv: -kv[1])[:3]
        out[int(c)] = {
            "profile": names[int(c)],
            "actions": acts or ["no distinctive elevated item; general provision applies"],
            "top_volunteered_themes": [{"theme": t, "pct": v} for t, v in top_themes],
        }
    return out

### `ml/supervised.py` - Optional supervised extension (methodology 5.7) and the incremental-value test.

In [ ]:
%%writefile ml/supervised.py
# -*- coding: utf-8 -*-
"""
Optional supervised extension (methodology 5.7) and the incremental-value test.

Two questions, both answered against an explicit baseline so a number like "44%
accuracy" cannot be mistaken for a result when the majority class is 33%:

  1. Are the discovered profiles predictable from information the clustering
     never saw (demographics, backlog, free-text themes)? If yes, the profiles
     correspond to something outside the twelve items. If no - which is the
     likely outcome given the weak structure - that is reported as the finding.

  2. Do free-text theme features add predictive value over the closed-ended
     items and demographics? This is the nested feature-set comparison the
     project's research question asks for.

Every score is a stratified 5-fold cross-validated mean with its standard
deviation, and a permutation test is run on the headline model so the gap over
baseline is checked against chance rather than assumed.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     cross_validate, permutation_test_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from . import config as C


def _cv(random_state=0):
    return StratifiedKFold(C.CV_FOLDS, shuffle=True, random_state=random_state)


def _models():
    return {
        "RandomForest": RandomForestClassifier(
            n_estimators=400, min_samples_leaf=3, random_state=0, n_jobs=-1),
        "LogisticRegression": make_pipeline(
            StandardScaler(with_mean=False),
            LogisticRegression(max_iter=2000)),
    }


def evaluate(X, y, model_name="RandomForest"):
    """Cross-validated accuracy and macro-F1 for one feature set."""
    clf = _models()[model_name]
    res = cross_validate(clf, X, y, cv=_cv(), scoring=("accuracy", "f1_macro"), n_jobs=1)
    return {
        "accuracy": round(float(res["test_accuracy"].mean()), 4),
        "accuracy_sd": round(float(res["test_accuracy"].std()), 4),
        "macro_f1": round(float(res["test_f1_macro"].mean()), 4),
        "macro_f1_sd": round(float(res["test_f1_macro"].std()), 4),
    }


def baseline(y):
    """Majority-class baseline, computed the same way as the models."""
    X = np.zeros((len(y), 1))
    dummy = DummyClassifier(strategy="most_frequent")
    acc = cross_val_score(dummy, X, y, cv=_cv(), scoring="accuracy").mean()
    return {"accuracy": round(float(acc), 4), "accuracy_sd": 0.0,
            "macro_f1": None, "macro_f1_sd": None}


def feature_set_comparison(feature_sets, y, label, model_name="RandomForest"):
    """Compare nested feature sets against the majority-class baseline."""
    rows = [{"features": "Majority-class baseline", "n_features": 0, **baseline(y)}]
    for name, X in feature_sets:
        rows.append({"features": name, "n_features": int(X.shape[1]),
                     **evaluate(X, y, model_name)})
    best = max(rows[1:], key=lambda r: r["accuracy"])
    lift = best["accuracy"] - rows[0]["accuracy"]
    return {
        "target": label,
        "model": model_name,
        "rows": rows,
        "best_feature_set": best["features"],
        "best_accuracy": best["accuracy"],
        "lift_over_baseline": round(float(lift), 4),
        "verdict": ("adds no usable predictive value over the majority-class baseline"
                    if lift < 0.03 else
                    "beats the majority-class baseline by %.1f points" % (100 * lift)),
    }


def permutation_check(X, y, model_name="RandomForest", n_permutations=200):
    """Permutation test: is the cross-validated score better than label chance?"""
    clf = _models()[model_name]
    score, perm_scores, p = permutation_test_score(
        clf, X, y, cv=_cv(), scoring="accuracy",
        n_permutations=n_permutations, random_state=C.RANDOM_STATE, n_jobs=1)
    return {
        "observed_accuracy": round(float(score), 4),
        "permuted_mean": round(float(np.mean(perm_scores)), 4),
        "permuted_p95": round(float(np.percentile(perm_scores, 95)), 4),
        "p_value": float(p),
        "n_permutations": int(n_permutations),
    }


def importances(X, y, model_name="RandomForest", top=15):
    """Permutation importance on a held-out split.

    Impurity importance is biased toward high-cardinality features, which matters
    here because department has fourteen levels; permutation importance on unseen
    data does not have that failure mode.
    """
    from sklearn.model_selection import train_test_split

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.25, random_state=C.RANDOM_STATE, stratify=y)
    clf = _models()[model_name].fit(Xtr, ytr)
    r = permutation_importance(clf, Xte, yte, n_repeats=15,
                               random_state=C.RANDOM_STATE, n_jobs=1)
    df = pd.DataFrame({
        "feature": list(X.columns),
        "importance": r.importances_mean.round(5),
        "sd": r.importances_std.round(5),
    }).sort_values("importance", ascending=False).set_index("feature")
    return df.head(top), round(float(clf.score(Xte, yte)), 4)


def profile_recovery(cluster_labels, demo, themes):
    """Can held-out information recover the cluster a student landed in?

    The clustering used only the twelve items. Demographics and text themes are
    genuinely external, so this is a real external-validity test rather than a
    re-description of the input.
    """
    y = pd.Series(cluster_labels).astype(str)
    sets = [
        ("Demographics only", demo),
        ("Text themes only", themes),
        ("Demographics + text themes", pd.concat([demo, themes], axis=1)),
    ]
    return feature_set_comparison(sets, y, "cluster profile")


def incremental_text_value(items, demo, themes, y, label):
    """Nested comparison: do text features add anything over items + demographics?"""
    sets = [
        ("Survey items only (12)", items),
        ("Text themes only (13)", themes),
        ("Items + demographics", pd.concat([items, demo], axis=1)),
        ("Items + demographics + text", pd.concat([items, demo, themes], axis=1)),
    ]
    report = feature_set_comparison(sets, y, label)
    rows = {r["features"]: r for r in report["rows"]}
    with_text = rows["Items + demographics + text"]["accuracy"]
    without = rows["Items + demographics"]["accuracy"]
    report["text_increment"] = round(float(with_text - without), 4)
    report["text_verdict"] = (
        "free-text theme features do not improve prediction over items + demographics"
        if with_text - without < 0.01 else
        "free-text theme features add %.1f accuracy points" % (100 * (with_text - without)))
    return report

### `ml/figures.py` - All plotting for the pipeline.

In [ ]:
%%writefile ml/figures.py
# -*- coding: utf-8 -*-
"""
All plotting for the pipeline.

One module so the visual language stays consistent: same palette, same grid
treatment, same left-aligned bold titles, direct value labels instead of a
legend wherever a legend would be redundant. Diverging colours are reserved for
Likert composition and correlations (which have a real midpoint); sequential
blue is used for magnitude heatmaps; the categorical ramp is only for series
identity.
"""
from __future__ import annotations

import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import dendrogram

from . import config as C
from . import lexicon as LX

plt.rcParams.update({
    "figure.facecolor": C.SURFACE, "axes.facecolor": C.SURFACE, "savefig.facecolor": C.SURFACE,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial"],
    "font.size": 9, "axes.titlesize": 10.5, "axes.labelsize": 9,
    "axes.edgecolor": C.BASE, "axes.linewidth": 0.8, "axes.labelcolor": C.INK2,
    "text.color": C.INK, "xtick.color": C.MUTED, "ytick.color": C.MUTED,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "grid.color": C.GRID, "grid.linewidth": 0.7,
    "legend.frameon": False, "legend.fontsize": 8.5,
    "figure.dpi": 170, "savefig.dpi": 170, "savefig.bbox": "tight",
})

SEQ_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list("seq", C.SEQ)
DIV_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list(
    "div", ["#b02f2f", "#e88a89", "#f0efec", "#86b6ef", "#1c5cab"])


def style(ax, xgrid=False, ygrid=True):
    ax.set_axisbelow(True)
    if ygrid:
        ax.grid(axis="y", visible=True, alpha=0.9)
    else:
        ax.grid(axis="y", visible=False)
    if xgrid:
        ax.grid(axis="x", visible=True, alpha=0.9)
        ax.grid(axis="y", visible=False)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(C.BASE)
    ax.tick_params(length=0)
    return ax


def save(fig, name):
    path = os.path.join(C.FIG_DIR, name)
    fig.savefig(path)
    plt.close(fig)
    return path


def _bare(ax):
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)


# --------------------------------------------------------------------------
# Descriptive
# --------------------------------------------------------------------------
def fig_profile(dists, n, name="fig01_respondent_profile.png"):
    fig, axes = plt.subplots(1, 4, figsize=(12.4, 2.9), gridspec_kw={"wspace": 0.62})
    for ax, (title, d) in zip(axes, dists):
        y = np.arange(len(d))[::-1]
        ax.barh(y, d["n"], height=0.62, color=C.CAT[0], edgecolor=C.SURFACE, linewidth=1.4)
        ax.set_yticks(y)
        ax.set_yticklabels(d.index, fontsize=8)
        ax.set_title(title, color=C.INK, pad=8, loc="left", fontweight="bold")
        for yy, (cnt, pct) in zip(y, d[["n", "pct"]].values):
            ax.text(cnt + n * 0.012, yy, "%d - %.0f%%" % (cnt, pct),
                    va="center", fontsize=7.6, color=C.INK2)
        ax.set_xlim(0, d["n"].max() * 1.62)
        ax.set_xticks([])
        style(ax, ygrid=False)
        ax.spines["bottom"].set_visible(False)
    fig.suptitle("Respondent profile  (n = %d)" % n, x=0.008, ha="left",
                 fontsize=11.5, fontweight="bold", color=C.INK, y=1.06)
    return save(fig, name)


def fig_department(d, name="fig02_department.png"):
    fig, ax = plt.subplots(figsize=(6.6, 3.6))
    y = np.arange(len(d))[::-1]
    ax.barh(y, d["n"], height=0.66, color=C.CAT[0], edgecolor=C.SURFACE, linewidth=1.4)
    ax.set_yticks(y)
    ax.set_yticklabels(d.index, fontsize=8.2)
    for yy, (cnt, pct) in zip(y, d[["n", "pct"]].values):
        ax.text(cnt + 4, yy, "%d (%.1f%%)" % (cnt, pct), va="center", fontsize=7.6, color=C.INK2)
    ax.set_xlim(0, d["n"].max() * 1.28)
    ax.set_xticks([])
    ax.set_title("Respondents by department", loc="left", color=C.INK, fontweight="bold", pad=8)
    style(ax, ygrid=False)
    ax.spines["bottom"].set_visible(False)
    return save(fig, name)


def fig_likert(raw, name="fig03_likert_composition.png"):
    """100% stacked diverging composition of the twelve items, raw scale."""
    order = raw[C.ITEMS].mean().sort_values().index.tolist()
    labels5 = ["1 Strongly disagree", "2 Disagree", "3 Neutral", "4 Agree", "5 Strongly agree"]
    fig, ax = plt.subplots(figsize=(9.6, 4.8))
    for i, it in enumerate(order):
        counts = raw[it].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
        pct = 100 * counts / counts.sum()
        left = 0.0
        for v in range(1, 6):
            w = pct[v]
            ax.barh(i, w, left=left, height=0.62, color=C.DIV5[v - 1],
                    edgecolor=C.SURFACE, linewidth=1.6)
            if w >= 7:
                ax.text(left + w / 2, i, "%.0f" % w, ha="center", va="center", fontsize=7.2,
                        color="#ffffff" if v in (1, 5) else C.INK)
            left += w
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(["%s  %s" % (C.ITEM_QNUM[i], C.ITEM_LABEL[i]) for i in order], fontsize=8.2)
    ax.set_xlim(0, 100)
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
    ax.set_title("Raw response composition of the twelve items  (items ordered by raw mean)",
                 loc="left", color=C.INK, fontweight="bold", pad=26)
    ax.legend(handles=[Patch(facecolor=C.DIV5[i], label=labels5[i]) for i in range(5)],
              loc="lower left", bbox_to_anchor=(0, 1.005), ncol=5, handlelength=1.1,
              handleheight=0.9, columnspacing=1.2)
    style(ax, xgrid=True, ygrid=False)
    return save(fig, name)


def fig_item_means(items, name="fig04_item_means.png"):
    means = items[C.ITEMS].mean().sort_values()
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    yy = np.arange(len(means))
    ax.hlines(yy, 1, means.values, color=C.GRID, linewidth=2)
    ax.plot(means.values, yy, "o", markersize=8, color=C.CAT[0],
            markeredgecolor=C.SURFACE, markeredgewidth=1.6, linestyle="none")
    for i, (nm, m) in enumerate(means.items()):
        ax.text(m + 0.07, i, "%.2f" % m, va="center", fontsize=7.8, color=C.INK2)
    ax.set_yticks(yy)
    ax.set_yticklabels([C.ITEM_LABEL[i] for i in means.index], fontsize=8.4)
    ax.set_xlim(1, 5.45)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xlabel("Mean after direction alignment  (higher = more strain / less support)")
    ax.set_title("Mean strain per item, all items aligned to a common direction",
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    style(ax, xgrid=True, ygrid=False)
    return save(fig, name)


def fig_correlation(struct, name="fig05_item_correlation.png"):
    Cm = pd.DataFrame(struct["correlation_matrix"]).loc[C.ITEMS, C.ITEMS]
    fig, ax = plt.subplots(figsize=(6.6, 5.6))
    im = ax.imshow(Cm.values, cmap=DIV_CMAP, vmin=-0.55, vmax=0.55)
    ax.set_xticks(range(len(C.ITEMS)))
    ax.set_xticklabels(C.ITEMS, rotation=55, ha="right", fontsize=7.6)
    ax.set_yticks(range(len(C.ITEMS)))
    ax.set_yticklabels(C.ITEMS, fontsize=7.6)
    for i in range(len(C.ITEMS)):
        for j in range(len(C.ITEMS)):
            v = Cm.values[i, j]
            if i != j and abs(v) >= 0.25:
                ax.text(j, i, "%.2f" % v, ha="center", va="center", fontsize=6.4,
                        color="#ffffff" if abs(v) > 0.42 else C.INK)
    ax.set_title("Inter-item correlation after direction alignment  (mean |r| = %.2f)"
                 % struct["mean_abs_interitem_r"],
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    cb = fig.colorbar(im, ax=ax, shrink=0.72, pad=0.02)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=7.5, length=0)
    _bare(ax)
    return save(fig, name)


# --------------------------------------------------------------------------
# PCA
# --------------------------------------------------------------------------
def fig_scree(pca_rep, name="fig06_scree_parallel.png"):
    eig = np.array(pca_rep["eigenvalues"])
    rand = np.array(pca_rep["parallel_analysis"]["random_p95"])
    xs = np.arange(1, len(eig) + 1)
    fig, ax = plt.subplots(figsize=(7.0, 3.4))
    keep = eig > rand
    ax.bar(xs, eig, width=0.6, color=[C.CAT[0] if k else C.BASE for k in keep],
           edgecolor=C.SURFACE, linewidth=1.4, label="Observed eigenvalue")
    ax.plot(xs, rand, "-o", color=C.CAT[1], linewidth=1.8, markersize=5,
            markeredgecolor=C.SURFACE, label="Random data, 95th percentile")
    ax.axhline(1.0, color=C.MUTED, linewidth=1.2, linestyle=":")
    ax.text(len(eig), 1.04, "Kaiser (eigenvalue = 1)", ha="right", fontsize=7.4, color=C.MUTED)
    for x, e in zip(xs, eig):
        ax.text(x, e + 0.05, "%.2f" % e, ha="center", fontsize=7, color=C.INK2)
    ax.set_xticks(xs)
    ax.set_xlabel("Principal component")
    ax.set_ylabel("Eigenvalue")
    ax.set_ylim(0, max(eig) * 1.25)
    ax.set_title("Parallel analysis retains %d components; Kaiser would retain %d"
                 % (pca_rep["parallel_analysis"]["n_retain"], pca_rep["n_kaiser"]),
                 loc="left", color=C.INK, fontweight="bold", pad=22)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.005), ncol=2)
    style(ax)
    return save(fig, name)


def fig_variance(pca_rep, name="fig07_explained_variance.png"):
    cum = np.array(pca_rep["cumulative_variance_pct"])
    xs = np.arange(1, len(cum) + 1)
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    ax.plot(xs, cum, "-o", color=C.CAT[0], linewidth=2, markersize=6,
            markeredgecolor=C.SURFACE, markeredgewidth=1.4)
    ax.axhline(95, color=C.CAT[1], linestyle="--", linewidth=1.4)
    ax.text(len(cum), 96, "95% target", ha="right", fontsize=7.6, color=C.CAT[1])
    for x, v in zip(xs, cum):
        ax.text(x, v - 5, "%.0f" % v, ha="center", fontsize=7, color=C.INK2)
    ax.set_xticks(xs)
    ax.set_xlabel("Components retained")
    ax.set_ylabel("Cumulative variance (%)")
    ax.set_ylim(0, 105)
    key = [k for k in pca_rep if k.startswith("n_for_")][0]
    ax.set_title("%d of 12 components are needed to reach 95%% of variance"
                 % pca_rep[key], loc="left", color=C.INK, fontweight="bold", pad=8)
    style(ax)
    return save(fig, name)


def fig_loadings(rot_df, name="fig08_rotated_loadings.png"):
    fig, ax = plt.subplots(figsize=(5.4 + 0.7 * rot_df.shape[1], 4.8))
    M = rot_df.loc[C.ITEMS].values
    im = ax.imshow(M, cmap=DIV_CMAP, vmin=-0.8, vmax=0.8, aspect="auto")
    ax.set_xticks(range(rot_df.shape[1]))
    ax.set_xticklabels(rot_df.columns, fontsize=8.5)
    ax.set_yticks(range(len(C.ITEMS)))
    ax.set_yticklabels([C.ITEM_LABEL[i] for i in C.ITEMS], fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if abs(M[i, j]) >= 0.30:
                ax.text(j, i, "%.2f" % M[i, j], ha="center", va="center", fontsize=7.2,
                        color="#ffffff" if abs(M[i, j]) > 0.6 else C.INK)
    ax.set_title("Varimax-rotated loadings  (values below |0.30| left blank)",
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    _bare(ax)
    return save(fig, name)


def fig_pca_scatter(scores, labels, names, sil, name="fig09_pca_scatter.png"):
    fig, ax = plt.subplots(figsize=(6.6, 5.0))
    for i, c in enumerate(sorted(set(labels))):
        m = labels == c
        ax.scatter(scores[m, 0], scores[m, 1], s=14, alpha=0.55, linewidths=0,
                   color=C.CAT[i % len(C.CAT)], label=names[int(c)])
    for i, c in enumerate(sorted(set(labels))):
        m = labels == c
        ax.scatter(scores[m, 0].mean(), scores[m, 1].mean(), s=170, marker="X",
                   color=C.CAT[i % len(C.CAT)], edgecolor=C.SURFACE, linewidth=2, zorder=5)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title("Clusters in PCA space overlap heavily  (silhouette = %.3f)" % sil,
                 loc="left", color=C.INK, fontweight="bold", pad=30)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.005), ncol=1, fontsize=8)
    style(ax, xgrid=True, ygrid=True)
    return save(fig, name)


def fig_tsne(emb, labels, names, name="fig10_tsne.png"):
    fig, ax = plt.subplots(figsize=(6.2, 5.0))
    for i, c in enumerate(sorted(set(labels))):
        m = labels == c
        ax.scatter(emb[m, 0], emb[m, 1], s=13, alpha=0.6, linewidths=0,
                   color=C.CAT[i % len(C.CAT)], label=names[int(c)])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("t-SNE projection - no separated islands appear",
                 loc="left", color=C.INK, fontweight="bold", pad=30)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.005), ncol=1, fontsize=8)
    _bare(ax)
    return save(fig, name)


# --------------------------------------------------------------------------
# Cluster selection & validation
# --------------------------------------------------------------------------
def fig_k_selection(km_table, gmm_table, knee_k, name="fig11_k_selection.png"):
    fig, axes = plt.subplots(1, 4, figsize=(13.2, 2.9), gridspec_kw={"wspace": 0.34})
    ks = list(km_table.index)

    ax = axes[0]
    ax.plot(ks, km_table["sse"], "-o", color=C.CAT[0], linewidth=2, markersize=6,
            markeredgecolor=C.SURFACE, markeredgewidth=1.4)
    ax.axvline(knee_k, color=C.CAT[1], linestyle="--", linewidth=1.4)
    ax.text(knee_k, km_table["sse"].max(), " knee k=%d" % knee_k, fontsize=7.6, color=C.CAT[1])
    ax.set_title("k-means SSE (elbow)", loc="left", color=C.INK, fontweight="bold", pad=8)
    ax.set_xlabel("k")

    ax = axes[1]
    ax.plot(ks, km_table["silhouette"], "-o", color=C.CAT[0], linewidth=2, markersize=6,
            markeredgecolor=C.SURFACE, markeredgewidth=1.4)
    ax.axhline(0.25, color=C.CAT[1], linestyle="--", linewidth=1.3)
    ax.text(ks[-1], 0.258, "weak-structure floor", ha="right", fontsize=7.2, color=C.CAT[1])
    for k, v in zip(ks, km_table["silhouette"]):
        ax.text(k, v + 0.006, "%.3f" % v, ha="center", fontsize=7, color=C.INK2)
    ax.set_ylim(0, max(0.30, float(km_table["silhouette"].max()) * 1.3))
    ax.set_title("Silhouette (higher better)", loc="left", color=C.INK, fontweight="bold", pad=8)
    ax.set_xlabel("k")

    ax = axes[2]
    ax.plot(ks, km_table["davies_bouldin"], "-o", color=C.CAT[2], linewidth=2, markersize=6,
            markeredgecolor=C.SURFACE, markeredgewidth=1.4)
    ax.set_title("Davies-Bouldin (lower better)", loc="left", color=C.INK, fontweight="bold", pad=8)
    ax.set_xlabel("k")

    ax = axes[3]
    ax.plot(ks, gmm_table["bic"], "-o", color=C.CAT[3], linewidth=2, markersize=6,
            markeredgecolor=C.SURFACE, markeredgewidth=1.4, label="BIC")
    ax.set_title("Gaussian-mixture BIC (lower better)", loc="left", color=C.INK,
                 fontweight="bold", pad=8)
    ax.set_xlabel("k")

    for ax in axes:
        ax.set_xticks(ks)
        style(ax)
    return save(fig, name)


def fig_gap(gap, name="fig11b_gap_statistic.png"):
    """Gap curve with Tibshirani error bars; k=1 included so 'no structure' is visible."""
    ks = gap["ks"]
    g = np.array(gap["gap"], dtype=float)
    sk = np.array(gap["s_k"], dtype=float)
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    ax.errorbar(ks, g, yerr=sk, fmt="-o", color=C.CAT[0], linewidth=2, markersize=6,
                markeredgecolor=C.SURFACE, markeredgewidth=1.4,
                ecolor=C.INK2, elinewidth=1, capsize=3)
    sel = gap["k_selected"]
    ax.scatter([sel], [g[ks.index(sel)]], s=170, marker="o", facecolor="none",
               edgecolor=C.CAT[1], linewidth=2, zorder=5)
    ax.annotate("selected k = %d" % sel, xy=(sel, g[ks.index(sel)]),
                xytext=(6, 10), textcoords="offset points", fontsize=8, color=C.CAT[1])
    ax.set_xticks(ks)
    ax.set_xlabel("k  (k = 1 means 'one homogeneous group')")
    ax.set_ylabel("Gap")
    ax.set_title("Gap statistic against a uniform null - %s"
                 % ("no structure found" if gap["supports_no_structure"]
                    else "structure found at k = %d" % sel),
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    style(ax)
    return save(fig, name)


def fig_dendrogram(link, k, name="fig12_dendrogram.png"):
    fig, ax = plt.subplots(figsize=(9.0, 3.6))
    dendrogram(link, ax=ax, no_labels=True, color_threshold=link[-(k - 1), 2],
               above_threshold_color=C.BASE)
    ax.set_ylabel("Merge distance (Ward)")
    ax.set_title("Ward dendrogram - merge cost rises smoothly, so no k is obviously correct",
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    style(ax)
    return save(fig, name)


def fig_silhouette_samples(sil_values, labels, names, name="fig13_silhouette_samples.png"):
    fig, ax = plt.subplots(figsize=(6.6, 4.2))
    y = 0
    ticks, ticklabels = [], []
    for i, c in enumerate(sorted(set(labels))):
        vals = np.sort(sil_values[labels == c])
        ax.fill_betweenx(np.arange(y, y + len(vals)), 0, vals,
                         color=C.CAT[i % len(C.CAT)], linewidth=0, alpha=0.85)
        ticks.append(y + len(vals) / 2)
        ticklabels.append(names[int(c)])
        y += len(vals) + 12
    ax.axvline(sil_values.mean(), color=C.INK, linestyle="--", linewidth=1.3)
    ax.text(sil_values.mean(), y, " mean %.3f" % sil_values.mean(), fontsize=7.6, color=C.INK)
    ax.axvline(0, color=C.MUTED, linewidth=1)
    ax.set_yticks(ticks)
    ax.set_yticklabels(ticklabels, fontsize=8)
    ax.set_xlabel("Silhouette coefficient")
    ax.set_title("Per-student silhouette: a large share of members sit near or below zero",
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    style(ax, xgrid=True, ygrid=False)
    return save(fig, name)


def fig_stability(boot, seeds, name="fig14_stability.png"):
    fig, ax = plt.subplots(figsize=(6.6, 2.6))
    bars = [("Bootstrap ARI (mean)", boot["ari_mean"]),
            ("Bootstrap ARI (5th pct)", boot["ari_p05"]),
            ("Seed-to-seed ARI (mean)", seeds["pairwise_ari_mean"]),
            ("Seed-to-seed ARI (min)", seeds["pairwise_ari_min"])]
    y = np.arange(len(bars))[::-1]
    vals = [b[1] for b in bars]
    colors = [C.CAT[0] if v >= 0.75 else (C.CAT[3] if v >= 0.5 else C.CAT[7]) for v in vals]
    ax.barh(y, vals, height=0.6, color=colors, edgecolor=C.SURFACE, linewidth=1.4)
    for yy, v in zip(y, vals):
        ax.text(v + 0.015, yy, "%.3f" % v, va="center", fontsize=8, color=C.INK2)
    ax.axvline(0.75, color=C.MUTED, linestyle="--", linewidth=1.2)
    ax.text(0.755, y.max() + 0.45, "0.75 = stable", fontsize=7.4, color=C.MUTED)
    ax.set_yticks(y)
    ax.set_yticklabels([b[0] for b in bars], fontsize=8.2)
    ax.set_xlim(0, 1.05)
    ax.set_title("Cluster stability under resampling and reseeding",
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    style(ax, xgrid=True, ygrid=False)
    return save(fig, name)


def fig_algorithm_agreement(M, name="fig15_algorithm_agreement.png"):
    fig, ax = plt.subplots(figsize=(5.6, 4.6))
    im = ax.imshow(M.values.astype(float), cmap=SEQ_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(len(M)))
    ax.set_xticklabels(M.columns, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(len(M)))
    ax.set_yticklabels(M.index, fontsize=8)
    for i in range(len(M)):
        for j in range(len(M)):
            v = float(M.values[i, j])
            ax.text(j, i, "%.2f" % v, ha="center", va="center", fontsize=7.6,
                    color="#ffffff" if v > 0.55 else C.INK)
    ax.set_title("Between-algorithm agreement (adjusted Rand)",
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    _bare(ax)
    return save(fig, name)


# --------------------------------------------------------------------------
# Profiles
# --------------------------------------------------------------------------
def fig_cluster_heatmap(Zp, names, name="fig16_cluster_profiles.png"):
    order = sorted(Zp.index, key=lambda c: Zp.loc[c].mean())
    M = Zp.loc[order, C.ITEMS].T.values
    fig, ax = plt.subplots(figsize=(2.4 + 1.5 * len(order), 4.8))
    lim = float(np.abs(M).max())
    im = ax.imshow(M, cmap=DIV_CMAP, vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([names[int(c)] for c in order], fontsize=8, rotation=16, ha="right")
    ax.set_yticks(range(len(C.ITEMS)))
    ax.set_yticklabels([C.ITEM_LABEL[i] for i in C.ITEMS], fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, "%+.2f" % M[i, j], ha="center", va="center", fontsize=7.4,
                    color="#ffffff" if abs(M[i, j]) > lim * 0.62 else C.INK)
    ax.set_title("Cluster item means as z-scores  (0 = sample average)",
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    cb = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=7.5, length=0)
    _bare(ax)
    return save(fig, name)


def fig_composition(comp, names, name="fig17_cluster_composition.png"):
    keys = [("year", "Academic year mix", C.YEAR_ORD),
            ("living", "Living arrangement mix", C.LIVE_ORD),
            ("cgpa", "CGPA band mix", C.CGPA_ORD)]
    keys = [k for k in keys if k[0] in comp]
    fig, axes = plt.subplots(1, len(keys), figsize=(4.9 * len(keys), 3.2))
    if len(keys) == 1:
        axes = [axes]
    for ax, (var, title, order) in zip(axes, keys):
        pct = comp[var]["pct"]
        clusters = sorted(pct)
        levels = [c for c in order if c in pct[clusters[0]]]
        ys = np.arange(len(clusters))[::-1]
        for ci, c in enumerate(clusters):
            left = 0.0
            for li, lv in enumerate(levels):
                v = pct[c][lv]
                ax.barh(ys[ci], v, left=left, height=0.6, color=C.CAT[li % len(C.CAT)],
                        edgecolor=C.SURFACE, linewidth=1.6)
                if v >= 9:
                    ax.text(left + v / 2, ys[ci], "%.0f" % v, ha="center", va="center",
                            fontsize=7.2, color="#ffffff" if li in (0, 5) else C.INK)
                left += v
        ax.set_yticks(ys)
        ax.set_yticklabels([names[int(c)] for c in clusters], fontsize=7.8)
        ax.set_xlim(0, 100)
        ax.set_xticks([0, 50, 100])
        ax.set_xticklabels(["0%", "50%", "100%"])
        ax.set_title("%s  (chi-square p = %.4f)" % (title, comp[var]["p"]),
                     loc="left", color=C.INK, fontweight="bold", pad=22)
        ax.legend(handles=[Patch(facecolor=C.CAT[i % len(C.CAT)], label=lv)
                           for i, lv in enumerate(levels)],
                  loc="lower left", bbox_to_anchor=(0, 1.005), ncol=len(levels),
                  handlelength=1.0, fontsize=7.6)
        style(ax, xgrid=True, ygrid=False)
    return save(fig, name)


def fig_discriminating(disc, name="fig18_discriminating_items.png"):
    d = disc.sort_values("eta_squared")
    fig, ax = plt.subplots(figsize=(7.0, 4.2))
    y = np.arange(len(d))
    ax.barh(y, d["eta_squared"] * 100, height=0.62, color=C.CAT[0],
            edgecolor=C.SURFACE, linewidth=1.4)
    for yy, v in zip(y, d["eta_squared"] * 100):
        ax.text(v + 0.4, yy, "%.1f%%" % v, va="center", fontsize=7.6, color=C.INK2)
    ax.set_yticks(y)
    ax.set_yticklabels([C.ITEM_LABEL[i] for i in d.index], fontsize=8.2)
    ax.set_xlabel("Variance in the item explained by the cluster split (eta-squared, %)")
    ax.set_xlim(0, float((d["eta_squared"] * 100).max()) * 1.25)
    ax.set_title("Which items actually separate the profiles",
                 loc="left", color=C.INK, fontweight="bold", pad=8)
    style(ax, xgrid=True, ygrid=False)
    return save(fig, name)


# --------------------------------------------------------------------------
# Text
# --------------------------------------------------------------------------
def fig_theme_prevalence(prev1, prev2, n1, n2, name="fig19_theme_prevalence.png"):
    order = prev1.index.tolist()
    fig, ax = plt.subplots(figsize=(8.2, 4.6))
    yy = np.arange(len(order))[::-1]
    h = 0.36
    ax.barh(yy + h / 2, [prev1[t] for t in order], height=h, color=C.CAT[0],
            edgecolor=C.SURFACE, linewidth=1.2, label="Current year (n=%d)" % n1)
    ax.barh(yy - h / 2, [prev2[t] for t in order], height=h, color=C.CAT[1],
            edgecolor=C.SURFACE, linewidth=1.2, label="Previous years (n=%d)" % n2)
    for i, t in enumerate(order):
        ax.text(prev1[t] + 0.4, yy[i] + h / 2, "%.0f%%" % prev1[t], va="center",
                fontsize=7.2, color=C.INK2)
        ax.text(prev2[t] + 0.4, yy[i] - h / 2, "%.0f%%" % prev2[t], va="center",
                fontsize=7.2, color=C.INK2)
    ax.set_yticks(yy)
    ax.set_yticklabels(order, fontsize=8.2)
    ax.set_xlabel("% of answering students who mentioned the theme")
    ax.set_xlim(0, max(prev1.max(), prev2.max()) * 1.22)
    ax.set_title("Stressor themes volunteered in the free-text answers",
                 loc="left", color=C.INK, fontweight="bold", pad=22)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.005), ncol=2)
    style(ax, xgrid=True, ygrid=False)
    return save(fig, name)


def fig_theme_heatmap(by_group, order_themes, title, name):
    keys = list(by_group)
    M = np.array([[by_group[g][t] for g in keys] for t in order_themes])
    fig, ax = plt.subplots(figsize=(2.6 + 1.1 * len(keys), 4.6))
    im = ax.imshow(M, cmap=SEQ_CMAP, aspect="auto", vmin=0, vmax=M.max())
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(["%s\n(n=%d)" % (g, by_group[g]["n"]) for g in keys], fontsize=8)
    ax.set_yticks(range(len(order_themes)))
    ax.set_yticklabels(order_themes, fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, "%.0f" % M[i, j], ha="center", va="center", fontsize=7.4,
                    color="#ffffff" if M[i, j] > M.max() * 0.55 else C.INK)
    ax.set_title(title, loc="left", color=C.INK, fontweight="bold", pad=8)
    _bare(ax)
    return save(fig, name)


def fig_theme_by_cluster(tbc, names, themes, name="fig22_theme_by_cluster.png"):
    clusters = sorted(tbc)
    x = np.arange(len(themes))
    w = 0.8 / len(clusters)
    fig, ax = plt.subplots(figsize=(8.2, 3.8))
    for ci, c in enumerate(clusters):
        vals = [tbc[c][t] for t in themes]
        off = (ci - (len(clusters) - 1) / 2) * w
        ax.bar(x + off, vals, width=w * 0.88, color=C.CAT[ci % len(C.CAT)],
               edgecolor=C.SURFACE, linewidth=1.2, label=names[int(c)])
        for xi, v in zip(x + off, vals):
            ax.text(xi, v + 0.3, "%.0f" % v, ha="center", fontsize=6.6, color=C.INK2)
    ax.set_xticks(x)
    ax.set_xticklabels([t.split(" &")[0].split(",")[0] for t in themes],
                       rotation=18, ha="right", fontsize=8)
    ax.set_ylabel("% mentioning the theme")
    ax.set_title("Volunteered themes by cluster - independent corroboration of the profiles",
                 loc="left", color=C.INK, fontweight="bold", pad=22)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.005), ncol=len(clusters), fontsize=7.8)
    style(ax)
    return save(fig, name)


# --------------------------------------------------------------------------
# Supervised
# --------------------------------------------------------------------------
def fig_model_comparison(report, name="fig23_model_comparison.png"):
    rows = report["rows"]
    base = rows[0]["accuracy"] * 100
    body = rows[1:]
    names_ = [r["features"] for r in body]
    acc = [r["accuracy"] * 100 for r in body]
    sd = [r["accuracy_sd"] * 100 for r in body]
    f1 = [(r["macro_f1"] or 0) * 100 for r in body]
    x = np.arange(len(body))
    w = 0.36
    fig, ax = plt.subplots(figsize=(1.9 * len(body) + 2.4, 3.4))
    ax.bar(x - w / 2, acc, width=w * 0.9, color=C.CAT[0], edgecolor=C.SURFACE,
           linewidth=1.2, label="Accuracy", yerr=sd, capsize=3,
           error_kw={"elinewidth": 1, "ecolor": C.INK2})
    ax.bar(x + w / 2, f1, width=w * 0.9, color=C.CAT[1], edgecolor=C.SURFACE,
           linewidth=1.2, label="Macro-F1")
    for xi, v in zip(x - w / 2, acc):
        ax.text(xi, v + 1.2, "%.1f" % v, ha="center", fontsize=7.4, color=C.INK2)
    for xi, v in zip(x + w / 2, f1):
        ax.text(xi, v + 1.2, "%.1f" % v, ha="center", fontsize=7.4, color=C.INK2)
    ax.axhline(base, color=C.CAT[7], linestyle="--", linewidth=1.5)
    ax.text(len(body) - 0.5, base + 1.4, "majority baseline %.1f%%" % base,
            ha="right", fontsize=7.6, color=C.CAT[7])
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(" + ", "\n+ ") for n in names_], fontsize=7.8)
    ax.set_ylabel("%")
    ax.set_ylim(0, max(max(acc), base) * 1.45)
    ax.set_title("Predicting %s - %s" % (report["target"], report["verdict"]),
                 loc="left", color=C.INK, fontweight="bold", pad=22)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.005), ncol=2)
    style(ax)
    return save(fig, name)

### `ml/embeddings.py` - Optional GPU branch: multilingual sentence embeddings over the free text.

In [ ]:
%%writefile ml/embeddings.py
# -*- coding: utf-8 -*-
"""
Optional GPU branch: multilingual sentence embeddings over the free text.

The methodology report *asserts* that transformer embeddings are unreliable on
answers this short and this code-mixed. That is a reasonable prior, but on
Kaggle a GPU is free, so the claim can be tested instead of assumed - which
turns a stated assumption into a measured result and strengthens the write-up
either way.

What this does:
  * embeds both free-text fields with a multilingual model (LaBSE by default,
    which covers Bangla script, unlike English-only sentence encoders);
  * clusters the embeddings with k-means and HDBSCAN;
  * scores those clusters against the frozen lexicon themes with adjusted Rand
    and normalised mutual information, and against answer length - because if
    embedding clusters mostly track *how long* an answer is rather than what it
    says, that is exactly the failure the report predicted.

Entirely optional. If sentence-transformers or a network connection is missing,
`run()` returns a skipped-status dict and the pipeline continues. Nothing
downstream depends on it.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config as C

DEFAULT_MODEL = "sentence-transformers/LaBSE"
FALLBACK_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def available():
    """True when sentence-transformers can be imported."""
    try:
        import sentence_transformers  # noqa: F401
        return True
    except Exception:
        return False


def _device():
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"


def embed(texts, model_name=DEFAULT_MODEL, batch_size=64):
    """Encode a list of strings; returns (matrix, info)."""
    from sentence_transformers import SentenceTransformer

    dev = _device()
    try:
        model = SentenceTransformer(model_name, device=dev)
        used = model_name
    except Exception:
        model = SentenceTransformer(FALLBACK_MODEL, device=dev)
        used = FALLBACK_MODEL

    X = model.encode(list(texts), batch_size=batch_size, show_progress_bar=False,
                     convert_to_numpy=True, normalize_embeddings=True)
    return X, {"model": used, "device": dev, "dim": int(X.shape[1])}


def run(texts, mask, theme_frame, k=None, model_name=DEFAULT_MODEL):
    """Embed, cluster, and compare against the lexicon. Never raises."""
    if not available():
        return {"status": "skipped",
                "reason": ("sentence-transformers is not installed. On Kaggle: turn Internet ON "
                           "in the notebook settings and run "
                           "`pip install -q sentence-transformers`.")}
    try:
        from sklearn.cluster import HDBSCAN, KMeans
        from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                                     silhouette_score)

        sel = mask.to_numpy()
        answers = texts.fillna("").astype(str)[sel].tolist()
        X, info = embed(answers, model_name=model_name)

        # Reference labelling from the frozen lexicon: the single dominant theme
        # per answer, so a hard partition can be compared against a hard partition.
        T = theme_frame[sel]
        dominant = T.to_numpy().argmax(axis=1)
        has_theme = T.to_numpy().sum(axis=1) > 0

        k = k or int(T.to_numpy().sum(axis=0).astype(bool).sum())
        k = max(2, min(k, 13))

        km = KMeans(n_clusters=k, n_init=20, random_state=C.RANDOM_STATE).fit(X)
        hdb = HDBSCAN(min_cluster_size=15).fit(X)

        lengths = np.array([len(a.split()) for a in answers], dtype=float)
        length_bins = pd.qcut(lengths, q=min(4, len(np.unique(lengths))),
                              labels=False, duplicates="drop")

        out = {
            "status": "ok",
            **info,
            "n_texts": int(len(answers)),
            "kmeans_k": int(k),
            "kmeans_silhouette": round(float(silhouette_score(X, km.labels_)), 4),
            "hdbscan_n_clusters": int(len(set(hdb.labels_)) - (1 if -1 in hdb.labels_ else 0)),
            "hdbscan_pct_noise": round(float(100 * (hdb.labels_ == -1).mean()), 1),
            "agreement_with_lexicon": {
                "kmeans_ari": round(float(adjusted_rand_score(dominant[has_theme],
                                                             km.labels_[has_theme])), 4),
                "kmeans_nmi": round(float(normalized_mutual_info_score(dominant[has_theme],
                                                                      km.labels_[has_theme])), 4),
                "hdbscan_ari": round(float(adjusted_rand_score(dominant[has_theme],
                                                              hdb.labels_[has_theme])), 4),
            },
            "confound_with_answer_length": {
                "kmeans_nmi_with_length_quartile": round(
                    float(normalized_mutual_info_score(length_bins, km.labels_)), 4),
                "note": ("If this is comparable to or larger than the agreement with the "
                         "lexicon, the embedding clusters are sorting answers by length "
                         "rather than by stressor - the failure mode the methodology "
                         "report predicted for text this short."),
            },
        }
        agree = out["agreement_with_lexicon"]["kmeans_nmi"]
        conf = out["confound_with_answer_length"]["kmeans_nmi_with_length_quartile"]
        out["verdict"] = (
            "embedding clusters recover the lexicon themes only weakly (NMI=%.2f) and track "
            "answer length about as strongly (NMI=%.2f); the lexicon remains the primary "
            "text method" % (agree, conf)
            if agree < 0.35 or conf >= agree else
            "embedding clusters recover the lexicon themes substantially (NMI=%.2f), so the "
            "transformer route is viable on this corpus after all" % agree)
        return out
    except Exception as exc:  # pragma: no cover - defensive by design
        return {"status": "failed", "reason": "%s: %s" % (type(exc).__name__, exc)}

### `ml/run_pipeline.py` - End-to-end pipeline: load -> audit -> structure -> PCA -> cluster -> validate

In [ ]:
%%writefile ml/run_pipeline.py
# -*- coding: utf-8 -*-
"""
End-to-end pipeline: load -> audit -> structure -> PCA -> cluster -> validate
-> text -> profile -> supervised -> persist.

Run locally:      python -m ml.run_pipeline
Run on Kaggle:    same command, or import and call `main()` from a notebook cell.

Everything it produces lands under `outputs/` (or /kaggle/working on Kaggle):
    results.json                  every number the report cites
    RESULTS_SUMMARY.md            human-readable narrative of this run
    figures/*.png                 all figures
    tables/*.csv                  every table, ready to paste into the report
    models/*.joblib               fitted scaler / PCA / k-means, plus a
                                  self-contained inference bundle
    student_level_assignments.csv per-student profile assignment
    stress_prepared.arff          the same prepared table, for WEKA
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time

import joblib
import numpy as np
import pandas as pd

from . import cluster as CL
from . import config as C
from . import dataio as IO
from . import embeddings as EMB
from . import figures as FIG
from . import lexicon as LX
from . import preprocess as PP
from . import profile as PR
from . import reduce as RD
from . import structure as ST
from . import supervised as SUP
from . import textmodel as TM
from . import validate as VA


def _log(msg):
    print(msg, flush=True)


def _banner(step, msg):
    _log("\n[%s] %s" % (step, msg))


def main(argv=None):
    ap = argparse.ArgumentParser(description="KUET academic-stress clustering pipeline")
    ap.add_argument("--data", default=None, help="path to the response workbook")
    ap.add_argument("--k", type=int, default=None,
                    help="force a cluster count instead of using the vote")
    ap.add_argument("--secondary-k", type=int, default=3,
                    help="also profile this k as a documented alternative (0 to disable)")
    ap.add_argument("--bootstrap", type=int, default=C.BOOTSTRAP_B,
                    help="number of bootstrap resamples for the stability check")
    ap.add_argument("--with-embeddings", action="store_true",
                    help="run the optional multilingual-embedding cross-check (needs GPU+internet)")
    ap.add_argument("--fast", action="store_true",
                    help="fewer resamples and permutations; for a quick smoke run")
    args = ap.parse_args(argv)

    if args.fast:
        args.bootstrap = min(args.bootstrap, 40)

    t0 = time.time()
    C.ensure_dirs()
    R = {"meta": {"pipeline_version": "1.0.0",
                  "random_state": C.RANDOM_STATE,
                  "on_kaggle": C.ON_KAGGLE,
                  "python": sys.version.split()[0]}}

    # ----------------------------------------------------------------- load
    _banner(1, "Loading and validating the export")
    df, path = IO.load_raw(args.data)
    R["meta"]["data_file"] = os.path.basename(path)
    schema = IO.validate_schema(df)
    names = IO.column_names(df)
    R["audit"] = IO.audit(df, names)
    n = len(df)
    _log("    %s  ->  %d responses x %d columns" % (os.path.basename(path), n, schema["n_columns"]))
    _log("    missing (closed-ended): %d | duplicate rows: %d | out-of-range Likert: %d"
         % (R["audit"]["closed_ended_missing"], R["audit"]["exact_duplicate_rows"],
            R["audit"]["likert_out_of_range"]))

    # -------------------------------------------------------- preprocessing
    _banner(2, "Preprocessing: direction alignment, encoding, standardisation")
    items, raw = PP.build_items(df, names)
    bg = PP.build_background(df, names)
    Z, scaler = PP.standardise(items)
    strain = PP.composite_score(items)
    R["response_style"] = IO.response_style(items)
    R["imbalance"] = PP.imbalance_report(bg)

    desc = PP.item_descriptives(items, raw)
    IO.save_table(desc, "t01_item_descriptives")
    R["item_descriptives"] = desc.reset_index().to_dict("records")

    dists = {
        "year": PP.distribution(bg["year"], C.YEAR_ORD),
        "cgpa": PP.distribution(bg["cgpa"], C.CGPA_ORD),
        "gender": PP.distribution(bg["gender"], C.GENDER_ORD),
        "living": PP.distribution(bg["living"], C.LIVE_ORD),
        "department": PP.distribution(bg["department"]),
    }
    R["demographics"] = {k: v.to_dict("index") for k, v in dists.items()}
    R["demographics"]["backlog_pct"] = round(float(100 * bg["backlog"].mean()), 1)
    for k, v in dists.items():
        IO.save_table(v, "t02_dist_%s" % k)
    _log("    reverse-coded %s; %d items standardised" % (", ".join(C.REVERSED), len(C.ITEMS)))

    # ---------------------------------------------------------- measurement
    _banner(3, "Measurement structure of the instrument")
    R["structure"] = ST.analyse(items)
    R["structure"]["verdict"] = ST.verdict(R["structure"])
    _log("    Cronbach alpha = %.3f | KMO = %.3f | mean |inter-item r| = %.3f"
         % (R["structure"]["cronbach_alpha_12"], R["structure"]["kmo"]["overall"],
            R["structure"]["mean_abs_interitem_r"]))
    _log("    %s" % R["structure"]["verdict"])
    IO.save_table(pd.DataFrame({
        "corrected_item_total": R["structure"]["corrected_item_total"],
        "alpha_if_deleted": R["structure"]["alpha_if_deleted"],
        "kmo_per_item": R["structure"]["kmo"]["per_item"],
    }), "t03_reliability")

    # ------------------------------------------------------------------ PCA
    _banner(4, "Dimensionality reduction (PCA + parallel analysis + varimax)")
    scores, pca_model, pca_rep, load_df, rot_df = RD.run_pca(Z)
    pca_rep["component_interpretation"] = RD.interpret_components(rot_df)
    R["pca"] = pca_rep
    IO.save_table(pd.DataFrame({
        "eigenvalue": pca_rep["eigenvalues"],
        "explained_pct": pca_rep["explained_variance_pct"],
        "cumulative_pct": pca_rep["cumulative_variance_pct"],
        "random_p95": pca_rep["parallel_analysis"]["random_p95"],
    }, index=["PC%d" % (i + 1) for i in range(len(pca_rep["eigenvalues"]))]), "t04_pca_variance")
    IO.save_table(rot_df, "t05_pca_loadings_varimax")
    var95 = [k for k in pca_rep if k.startswith("n_for_")][0]
    _log("    %d components reach 95%% of variance; parallel analysis retains %d; "
         "PC1+PC2 = %.1f%%" % (pca_rep[var95], pca_rep["parallel_analysis"]["n_retain"],
                               pca_rep["variance_at_2_components_pct"]))

    # ------------------------------------------------------------- clustering
    _banner(5, "Clustering: k-means / Gaussian mixture / hierarchical / Gower")
    km_models, km_table, km_rep = CL.kmeans_sweep(Z)
    IO.save_table(km_table, "t06_kmeans_sweep")
    _log("    k-means sweep done  (SSE knee at k=%d, best silhouette at k=%d)"
         % (km_rep["sse_knee_k"], km_rep["best_silhouette_k"]))

    gmm_models, gmm_table, gmm_rep = CL.gmm_sweep(Z)
    IO.save_table(gmm_table, "t07_gmm_sweep")
    _log("    Gaussian mixture done  (BIC picks k=%d, CV log-likelihood picks k=%d)"
         % (gmm_rep["best_bic_k"], gmm_rep["best_cv_loglik_k"]))

    links, hier_labels, hier_rep = CL.hierarchical(Z)
    _log("    hierarchical done  -> %s" % hier_rep["note"])
    if hier_rep["degenerate_methods"]:
        _log("    degenerate linkages excluded from the cut: %s"
             % ", ".join(hier_rep["degenerate_methods"]))

    gap = CL.gap_statistic(Z, n_refs=10 if args.fast else 25)
    _log("    gap statistic: %s" % gap["interpretation"])

    selection = CL.select_k(km_rep, gmm_rep, hier_rep, gap_report=gap, km_table=km_table)
    K = args.k or selection["k_selected"]
    selection["k_used"] = int(K)
    selection["k_forced"] = args.k is not None
    R["k_selection"] = selection
    _log("    k selected = %d  (%d of %d signals agree, %s; votes: %s)"
         % (K, selection["n_agreeing"], selection["n_signals"],
            selection["tie_break"], selection["votes"]))

    km = km_models[K] if K in km_models else CL.KMeans(
        n_clusters=K, n_init=C.N_INIT, random_state=C.RANDOM_STATE).fit(Z)
    labels = km.labels_

    # Mixed-data cross-check over items + background, via Gower distance.
    mixed = pd.concat([items[C.ITEMS], bg[["year_ord", "cgpa_ord", "backlog"]],
                       bg[["gender", "living"]]], axis=1)
    D = CL.gower_matrix(mixed, numeric_cols=C.ITEMS + ["year_ord", "cgpa_ord"],
                        categorical_cols=["gender", "living", "backlog"])
    gower_labels_by, gower_table, gower_rep = CL.gower_sweep(D, ks=C.K_RANGE)
    IO.save_table(gower_table, "t08_gower_kmedoids_sweep")
    R["gower"] = gower_rep
    _log("    Gower k-medoids done  (best silhouette at k=%d)" % gower_rep["best_silhouette_k"])

    R["clustering"] = {"kmeans": km_rep, "gmm": gmm_rep, "hierarchical": hier_rep}

    # ------------------------------------------------------------ validation
    _banner(6, "Validation: internal indices, stability, consensus, external checks")
    sil_report, sil_values = VA.silhouette_breakdown(Z, labels)
    R["validation"] = {"silhouette": sil_report}
    R["validation"]["structure_verdict"] = CL.structure_verdict(sil_report["overall_mean"])
    _log("    silhouette = %.4f -> %s"
         % (sil_report["overall_mean"], R["validation"]["structure_verdict"]))

    boot = VA.bootstrap_stability(Z, K, labels, b=args.bootstrap)
    seeds = VA.seed_stability(Z, K)
    R["validation"]["bootstrap_stability"] = boot
    R["validation"]["seed_stability"] = seeds
    _log("    bootstrap ARI = %.3f (5th pct %.3f) | seed-to-seed ARI = %.3f"
         % (boot["ari_mean"], boot["ari_p05"], seeds["pairwise_ari_mean"]))

    M = VA.consensus(Z, K, b=40 if args.fast else 100)
    R["validation"]["consensus_by_cluster"] = VA.consensus_by_cluster(M, labels)

    R["validation"]["classes_to_clusters"] = VA.classes_to_clusters(labels, bg)
    for var, d in R["validation"]["classes_to_clusters"].items():
        _log("    external | %-11s chi2 p=%.4g  Cramer's V=%.3f  ARI=%.3f"
             % (var, d["p"], d["cramers_v"], d["adjusted_rand"]))

    algo_labels = {
        "k-means": labels,
        "GMM": gmm_models[K].predict(Z),
        "Ward": hier_labels["ward"][K],
        "Average-linkage": hier_labels["average"][K],
        "Spectral": CL.spectral_labels(Z, K),
        "Gower k-medoids": gower_labels_by[K],
    }
    agree = VA.compare_algorithms(algo_labels)
    IO.save_table(agree, "t09_algorithm_agreement")
    R["validation"]["algorithm_agreement"] = agree.to_dict()
    _log("    between-algorithm ARI vs k-means: %s"
         % {k: float(agree.loc["k-means", k]) for k in agree.columns if k != "k-means"})

    # ------------------------------------------------------------------ text
    _banner(7, "Open-ended text: lexicon, topic-model cross-check, null tests")
    txt1 = df[names["open_current"]]
    txt2 = df[names["open_previous"]]
    T1, mask1, _ = LX.tag_frame(txt1)
    T2, mask2, _ = LX.tag_frame(txt2)

    R["text"] = {
        "lexicon_version": LX.LEXICON_VERSION,
        "profile_current": LX.text_profile(txt1, mask1),
        "profile_previous": LX.text_profile(txt2, mask2),
        "coverage_current": LX.coverage(T1, mask1),
        "coverage_previous": LX.coverage(T2, mask2),
    }
    prev1 = LX.prevalence(T1, mask1)
    prev2 = LX.prevalence(T2, mask2)
    R["text"]["prevalence_current"] = prev1.to_dict()
    R["text"]["prevalence_previous"] = prev2.to_dict()
    IO.save_table(pd.DataFrame({"current_year_pct": prev1, "previous_years_pct": prev2}),
                  "t10_theme_prevalence")
    IO.save_table(LX.export_patterns(), "t11_lexicon_patterns")
    _log("    lexicon covers %.1f%% of current-year answers (%.2f themes each), %.1f%% of previous"
         % (R["text"]["coverage_current"]["coverage_pct"],
            R["text"]["coverage_current"]["mean_themes_per_answer"],
            R["text"]["coverage_previous"]["coverage_pct"]))

    answered_txt = txt1.fillna("").astype(str)[mask1.to_numpy()].tolist()
    topic_rep, _ = TM.topic_model_report(answered_txt)
    R["text"]["topic_model"] = topic_rep
    R["text"]["top_terms"] = TM.top_terms(answered_txt)

    R["text"]["null_checks"] = TM.null_checks(strain.to_numpy(), txt1, mask1)
    _log("    null check | non-response: %s" % R["text"]["null_checks"]["non_response"]["verdict"])
    _log("    null check | answer length: %s" % R["text"]["null_checks"]["answer_length"]["verdict"])

    R["text"]["theme_strain_association"] = TM.theme_strain_association(T1, mask1, strain.to_numpy())
    R["text"]["theme_by_year"] = TM.theme_by_group(T1, mask1, bg["year"])
    R["text"]["theme_by_gender"] = TM.theme_by_group(T1, mask1, bg["gender"])
    R["text"]["theme_by_living"] = TM.theme_by_group(T1, mask1, bg["living"])

    # --------------------------------------------------------------- profiles
    _banner(8, "Cluster profiling and naming")
    cnames, cdetail, Zp = PR.name_clusters(items, labels)
    R["profiles"] = {"names": {str(k): v for k, v in cnames.items()}, "detail": cdetail}
    prof = PR.profile_table(items, labels)
    IO.save_table(prof, "t12_cluster_item_means")
    IO.save_table(Zp, "t13_cluster_item_zscores")
    R["profiles"]["item_means"] = prof.to_dict("index")
    R["profiles"]["item_zscores"] = Zp.to_dict("index")

    summary = PR.cluster_summary(items, labels, bg, cnames)
    IO.save_table(summary, "t14_cluster_summary")
    R["profiles"]["summary"] = summary.reset_index().to_dict("records")
    for _, r in summary.iterrows():
        _log("    %-52s n=%3d (%4.1f%%)  strain=%.2f" %
             (r["profile"], r["n"], r["pct_of_sample"], r["mean_strain"]))

    R["profiles"]["composition"] = PR.composition(labels, bg)
    tbc = PR.theme_by_cluster(T1, mask1, labels)
    R["profiles"]["theme_by_cluster"] = tbc

    disc, top_items = PR.discriminating_items(items, labels)
    IO.save_table(disc, "t15_discriminating_items")
    R["profiles"]["discriminating_items"] = disc.reset_index().to_dict("records")
    R["profiles"]["recommendations"] = PR.recommendations(cnames, cdetail, tbc)
    _log("    most discriminating items: %s" % ", ".join(top_items))

    # A second solution at the k the project proposal anticipated, so the report
    # can show what is gained or lost by splitting further. Reported as an
    # alternative, never as the headline: its silhouette is the honest cost.
    alt_fig = None
    if args.secondary_k and args.secondary_k != K:
        k2 = args.secondary_k
        km2 = CL.KMeans(n_clusters=k2, n_init=C.N_INIT, random_state=C.RANDOM_STATE).fit(Z)
        n2, d2, Zp2 = PR.name_clusters(items, km2.labels_)
        sil2, _ = VA.silhouette_breakdown(Z, km2.labels_)
        sum2 = PR.cluster_summary(items, km2.labels_, bg, n2)
        IO.save_table(sum2, "t17_alternative_k%d_summary" % k2)
        IO.save_table(Zp2, "t18_alternative_k%d_zscores" % k2)
        R["alternative_solution"] = {
            "k": int(k2),
            "silhouette": sil2["overall_mean"],
            "silhouette_at_selected_k": sil_report["overall_mean"],
            "names": {str(a): b for a, b in n2.items()},
            "summary": sum2.reset_index().to_dict("records"),
            "agreement_with_selected": round(float(
                VA.adjusted_rand_score(labels, km2.labels_)), 4),
            "note": ("Provided because the project proposal anticipated roughly three "
                     "personas. It is an alternative segmentation of the same continuum, "
                     "not a better-supported one: silhouette %.4f vs %.4f at k=%d."
                     % (sil2["overall_mean"], sil_report["overall_mean"], K)),
        }
        alt_fig = FIG.fig_cluster_heatmap(Zp2, n2, "fig25_alternative_k%d_profiles.png" % k2)
        _log("    alternative k=%d profiled (silhouette %.4f vs %.4f at k=%d)"
             % (k2, sil2["overall_mean"], sil_report["overall_mean"], K))

    # ------------------------------------------------------------- supervised
    _banner(9, "Supervised extension and incremental-value tests")
    demo = PP.encode_background(bg)
    R["supervised"] = {}
    R["supervised"]["profile_recovery"] = SUP.profile_recovery(labels, demo, T1)
    _log("    recovering the profile from held-out features: %s"
         % R["supervised"]["profile_recovery"]["verdict"])

    # The CGPA band is the target here, so its ordinal encoding must leave the
    # feature matrix - otherwise the model simply reads the answer off an input.
    demo_no_cgpa = PP.encode_background(bg, exclude=("cgpa_ord",))
    y_cgpa = bg["cgpa"].astype(str)
    R["supervised"]["cgpa_prediction"] = SUP.incremental_text_value(
        items[C.ITEMS], demo_no_cgpa, T1, y_cgpa, "CGPA band")
    _log("    predicting CGPA band: %s" % R["supervised"]["cgpa_prediction"]["verdict"])
    _log("    text increment: %s" % R["supervised"]["cgpa_prediction"]["text_verdict"])

    if not args.fast:
        R["supervised"]["permutation_check_cgpa"] = SUP.permutation_check(
            pd.concat([items[C.ITEMS], demo_no_cgpa], axis=1), y_cgpa, n_permutations=100)

    imp, held = SUP.importances(pd.concat([items[C.ITEMS], demo_no_cgpa, T1], axis=1), y_cgpa)
    IO.save_table(imp, "t16_permutation_importance")
    R["supervised"]["top_features"] = imp.reset_index().to_dict("records")

    # ------------------------------------------------------------- embeddings
    if args.with_embeddings:
        _banner(10, "Optional: multilingual sentence-embedding cross-check")
        R["embeddings"] = EMB.run(txt1, mask1, T1)
        _log("    %s" % R["embeddings"].get("verdict", R["embeddings"].get("reason", "")))
    else:
        R["embeddings"] = {"status": "not requested",
                           "reason": "run with --with-embeddings (needs internet + ideally a GPU)"}

    # ---------------------------------------------------------------- figures
    _banner(11, "Rendering figures")
    figs = []
    figs.append(FIG.fig_profile(
        [("Academic year", dists["year"][dists["year"]["n"] >= 3]),
         ("Current CGPA", dists["cgpa"]), ("Gender", dists["gender"]),
         ("Living arrangement", dists["living"])], n))
    figs.append(FIG.fig_department(dists["department"]))
    figs.append(FIG.fig_likert(raw))
    figs.append(FIG.fig_item_means(items))
    figs.append(FIG.fig_correlation(R["structure"]))
    figs.append(FIG.fig_scree(pca_rep))
    figs.append(FIG.fig_variance(pca_rep))
    figs.append(FIG.fig_loadings(rot_df))
    figs.append(FIG.fig_pca_scatter(scores, labels, cnames, sil_report["overall_mean"]))
    try:
        figs.append(FIG.fig_tsne(RD.tsne_projection(Z), labels, cnames))
    except Exception as exc:
        _log("    t-SNE skipped: %s" % exc)
    figs.append(FIG.fig_k_selection(km_table, gmm_table, km_rep["sse_knee_k"]))
    figs.append(FIG.fig_gap(gap))
    figs.append(FIG.fig_dendrogram(links["ward"], K))
    figs.append(FIG.fig_silhouette_samples(sil_values, labels, cnames))
    figs.append(FIG.fig_stability(boot, seeds))
    figs.append(FIG.fig_algorithm_agreement(agree))
    figs.append(FIG.fig_cluster_heatmap(Zp, cnames))
    figs.append(FIG.fig_composition(R["profiles"]["composition"], cnames))
    figs.append(FIG.fig_discriminating(disc))
    figs.append(FIG.fig_theme_prevalence(prev1, prev2, int(mask1.sum()), int(mask2.sum())))
    order_t = prev1.index.tolist()
    if R["text"]["theme_by_year"]:
        figs.append(FIG.fig_theme_heatmap(R["text"]["theme_by_year"], order_t,
                                          "Theme prevalence (%) by academic year",
                                          "fig20_theme_by_year.png"))
    if R["text"]["theme_by_living"]:
        figs.append(FIG.fig_theme_heatmap(R["text"]["theme_by_living"], order_t,
                                          "Theme prevalence (%) by living arrangement",
                                          "fig21_theme_by_living.png"))
    figs.append(FIG.fig_theme_by_cluster(tbc, cnames, order_t[:6]))
    figs.append(FIG.fig_model_comparison(R["supervised"]["cgpa_prediction"]))
    figs.append(FIG.fig_model_comparison(R["supervised"]["profile_recovery"],
                                         "fig24_profile_recovery.png"))
    if alt_fig:
        figs.append(alt_fig)   # rendered earlier, alongside the alternative-k profiling
    _log("    %d figures written to %s" % (len(figs), C.FIG_DIR))
    R["figures"] = [os.path.basename(f) for f in figs]

    # ---------------------------------------------------------------- persist
    _banner(12, "Persisting models, tables and the run report")

    out = pd.DataFrame({
        "timestamp": df[names["timestamp"]],
        "year": bg["year"], "cgpa": bg["cgpa"], "gender": bg["gender"],
        "living": bg["living"], "department": bg["department"], "backlog": bg["backlog"],
    })
    for it in C.ITEMS:
        out["item_" + it] = items[it]
    out["strain_index"] = strain.round(3)
    out["cluster"] = labels
    out["profile"] = [cnames[int(c)] for c in labels]
    out["silhouette"] = np.round(sil_values, 4)
    out["pc1"] = np.round(scores[:, 0], 4)
    out["pc2"] = np.round(scores[:, 1], 4)
    for t in LX.THEME_NAMES:
        out["theme_" + t.split(" ")[0].strip("&,").lower()] = T1[t]
    assign_path = os.path.join(C.OUT_DIR, "student_level_assignments.csv")
    out.to_csv(assign_path, index=False, encoding="utf-8-sig")

    bundle = {
        "version": "1.0.0",
        "scaler": scaler,
        "pca": pca_model,
        "kmeans": km,
        "items": C.ITEMS,
        "reversed_items": C.REVERSED,
        "cluster_names": {int(k): v for k, v in cnames.items()},
        "k": int(K),
        "silhouette": sil_report["overall_mean"],
        "trained_on": {"n": int(n), "file": os.path.basename(path)},
    }
    joblib.dump(bundle, os.path.join(C.MODEL_DIR, "stress_profile_model.joblib"))
    joblib.dump(gmm_models[K], os.path.join(C.MODEL_DIR, "gaussian_mixture_k%d.joblib" % K))

    arff_df = pd.concat([items[C.ITEMS], bg[["year", "cgpa", "gender", "living", "department"]],
                         bg["backlog"].map({0: "No", 1: "Yes"}).rename("backlog"),
                         T1.rename(columns=lambda c: "theme_" + c).replace({0: "no", 1: "yes"}),
                         pd.Series([cnames[int(c)] for c in labels], name="profile",
                                   index=items.index)], axis=1)
    IO.write_arff(
        arff_df, os.path.join(C.OUT_DIR, "stress_prepared.arff"),
        relation="kuet_academic_stress",
        numeric=C.ITEMS,
        nominal={
            "year": C.YEAR_ORD, "cgpa": C.CGPA_ORD, "gender": C.GENDER_ORD,
            "living": C.LIVE_ORD,
            "department": sorted(bg["department"].unique()),
            "backlog": ["No", "Yes"],
            "profile": [cnames[int(c)] for c in sorted(set(labels))],
            **{"theme_" + t: ["no", "yes"] for t in LX.THEME_NAMES},
        },
        string_cols=[])

    R["meta"]["runtime_seconds"] = round(time.time() - t0, 1)
    IO.save_json(R, os.path.join(C.OUT_DIR, "results.json"))
    write_summary(R, os.path.join(C.OUT_DIR, "RESULTS_SUMMARY.md"))

    _log("\nDone in %.1fs. Outputs -> %s" % (R["meta"]["runtime_seconds"], C.OUT_DIR))
    _log("  results.json, RESULTS_SUMMARY.md, student_level_assignments.csv, stress_prepared.arff")
    _log("  %d figures, %d tables, %d model files"
         % (len(figs), len(os.listdir(C.TAB_DIR)), len(os.listdir(C.MODEL_DIR))))
    return R


# --------------------------------------------------------------------------
# Narrative summary
# --------------------------------------------------------------------------
def write_summary(R, path):
    """Write a markdown summary whose sentences are generated from the numbers.

    Deliberately templated: every claim below is interpolated from `results.json`,
    so the narrative cannot drift away from what the run actually produced.
    """
    s, v, k = R["structure"], R["validation"], R["k_selection"]
    txt = R["text"]
    L = []
    A = L.append

    A("# Run summary - KUET academic-stress clustering\n")
    A("Pipeline %s | data `%s` | n = %d | random_state = %d | %.1f s\n"
      % (R["meta"]["pipeline_version"], R["meta"]["data_file"], R["audit"]["n_rows"],
         R["meta"]["random_state"], R["meta"]["runtime_seconds"]))

    A("\n## 1. Data integrity\n")
    a = R["audit"]
    A("- %d responses, %d closed-ended missing values, %d exact duplicate rows, "
      "%d out-of-range Likert values." % (a["n_rows"], a["closed_ended_missing"],
                                          a["exact_duplicate_rows"], a["likert_out_of_range"]))
    A("- Collected %s to %s (%d days), peak %d responses on %s."
      % (a["collection_start"][:10], a["collection_end"][:10], a["collection_days"],
         a["peak_day_n"], a["peak_day"]))
    rs = R["response_style"]
    A("- Straight-lining %.1f%% of respondents; %.1f%% used all five scale points; "
      "mean within-row SD %.2f." % (rs["straight_lining_pct"], rs["pct_using_all_five_points"],
                                    rs["mean_within_row_sd"]))

    A("\n## 2. Does the instrument measure one thing?\n")
    A("- Cronbach's alpha = **%.3f** across twelve items (%.3f on the %d-item core)."
      % (s["cronbach_alpha_12"], s["cronbach_alpha_core"], len(s["core_items"])))
    A("- KMO = %.3f; Bartlett chi-square = %.1f, p = %.3g; mean |inter-item r| = %.3f."
      % (s["kmo"]["overall"], s["bartlett"]["chi2"], s["bartlett"]["p"],
         s["mean_abs_interitem_r"]))
    A("- Weakest items: %s." % (", ".join(s["weak_items"]) or "none"))
    A("\n> %s\n" % s["verdict"])

    A("\n## 3. Dimensionality\n")
    p = R["pca"]
    key = [x for x in p if x.startswith("n_for_")][0]
    A("- %d of 12 components are needed for 95%% of variance; PC1+PC2 explain only %.1f%%."
      % (p[key], p["variance_at_2_components_pct"]))
    A("- Kaiser would retain %d components; parallel analysis against random data retains %d."
      % (p["n_kaiser"], p["parallel_analysis"]["n_retain"]))
    A("- Because variance is spread almost evenly, PCA is used for visualisation and "
      "structure description, not to compress the feature set before clustering.")

    A("\n## 4. How many clusters?\n")
    A("| signal | k |")
    A("|---|---|")
    for name, kk in k["votes"].items():
        A("| %s | %d |" % (name.replace("_", " "), kk))
    A("\n- **k = %d** selected (%d of %d signals agree)%s."
      % (k["k_used"], k["n_agreeing"], k["n_signals"],
         "; forced by --k" if k["k_forced"] else ""))

    A("\n## 5. Are the clusters real?\n")
    sil = v["silhouette"]
    A("- Silhouette = **%.4f** -> %s." % (sil["overall_mean"], v["structure_verdict"]))
    A("- %.1f%% of students have a negative silhouette (closer to another cluster than their own)."
      % sil["pct_negative_overall"])
    b, sd = v["bootstrap_stability"], v["seed_stability"]
    A("- Bootstrap ARI = %.3f (5th percentile %.3f) over %d resamples; seed-to-seed ARI = %.3f."
      % (b["ari_mean"], b["ari_p05"], b["n_resamples"], sd["pairwise_ari_mean"]))
    A("- Held-out background variables (none of which entered the model):")
    for var, d in v["classes_to_clusters"].items():
        A("  - %s: chi-square p = %.3g, Cramer's V = %.3f, ARI = %.3f"
          % (var, d["p"], d["cramers_v"], d["adjusted_rand"]))

    A("\n## 6. The profiles\n")
    A("| profile | n | % of sample | mean strain | modal year | modal living | % female |")
    A("|---|---|---|---|---|---|---|")
    for r in R["profiles"]["summary"]:
        A("| %s | %d | %.1f | %.2f | %s | %s | %.1f |"
          % (r["profile"], r["n"], r["pct_of_sample"], r["mean_strain"],
             r["modal_year"], r["modal_living"], r["pct_female"]))
    A("\nMost discriminating items (eta-squared):")
    for r in R["profiles"]["discriminating_items"][:5]:
        A("- %s: %.3f" % (r["label"], r["eta_squared"]))

    A("\n## 7. Free-text findings\n")
    tc = txt["coverage_current"]
    tp = txt["profile_current"]
    A("- %d of %d students answered the current-stressor question (%.1f%%); median %g words, "
      "%.1f%% two words or fewer." % (tp["n_answered"], R["audit"]["n_rows"],
                                      tp["response_rate_pct"], tp["words_median"],
                                      tp["pct_le2_words"]))
    A("- Script mix: %.1f%% Latin, %.1f%% Bangla, %.1f%% mixed, plus %d romanised-Bangla answers."
      % (tp["lang_latin_pct"], tp["lang_bangla_pct"], tp["lang_mixed_pct"], tp["romanised_n"]))
    A("- The frozen lexicon (v%s, 13 themes) tags %.1f%% of answers, %.2f themes each."
      % (txt["lexicon_version"], tc["coverage_pct"], tc["mean_themes_per_answer"]))
    top3 = list(txt["prevalence_current"].items())[:3]
    A("- Most volunteered themes: %s." % ", ".join("%s (%.0f%%)" % (t, p) for t, p in top3))
    nc = txt["null_checks"]
    A("- Pre-specified null check 1 (non-response): %s (p = %.3g, d = %.2f)."
      % (nc["non_response"]["verdict"], nc["non_response"]["p"], nc["non_response"]["cohens_d"]))
    A("- Pre-specified null check 2 (answer length): %s (rho = %.3f, p = %.3g)."
      % (nc["answer_length"]["verdict"], nc["answer_length"]["spearman_rho"],
         nc["answer_length"]["p"]))

    A("\n## 8. Supervised checks\n")
    pr = R["supervised"]["profile_recovery"]
    cg = R["supervised"]["cgpa_prediction"]
    A("- Recovering the cluster from held-out features: best %.3f vs %.3f baseline - %s."
      % (pr["best_accuracy"], pr["rows"][0]["accuracy"], pr["verdict"]))
    A("- Predicting CGPA band: best %.3f vs %.3f baseline - %s."
      % (cg["best_accuracy"], cg["rows"][0]["accuracy"], cg["verdict"]))
    A("- %s." % cg["text_verdict"])

    emb = R.get("embeddings", {})
    if emb.get("status") == "ok":
        A("\n## 9. Embedding cross-check\n")
        A("- Model %s on %s, %d texts." % (emb["model"], emb["device"], emb["n_texts"]))
        A("- %s" % emb["verdict"])

    A("\n## Headline\n")
    A("> The twelve items are a checklist of partly independent stressors rather than one "
      "scale (alpha = %.2f). Students do not fall into naturally separated groups "
      "(silhouette = %.3f); the k = %d partition is a **useful segmentation of a continuum**, "
      "reported as such. What the free text adds is *what* the strain is about, and the "
      "lexicon covers %.0f%% of answers including the Bangla-script ones that an "
      "English-only pipeline would drop."
      % (s["cronbach_alpha_12"], sil["overall_mean"], k["k_used"], tc["coverage_pct"]))

    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(L) + "\n")
    return path


if __name__ == "__main__":
    main()

## 2. Run the full pipeline

Roughly 3-6 minutes on a Kaggle CPU instance. Add `--fast` for a ~2 minute smoke run
with fewer resamples.

Everything lands in `/kaggle/working`: `results.json`, `RESULTS_SUMMARY.md`, the
figures, the tables, the fitted models, the per-student assignments and the ARFF
export for WEKA.

In [ ]:
import importlib
import ml.run_pipeline as RP
importlib.reload(RP)

R = RP.main([])          # add "--fast" for a quick run, "--k", "3" to force k

## 3. Run summary

In [ ]:
from IPython.display import Markdown, display
import ml.config as C

with open(os.path.join(C.OUT_DIR, "RESULTS_SUMMARY.md"), encoding="utf-8") as fh:
    display(Markdown(fh.read()))

## 4. Figures

In [ ]:
from IPython.display import Image, display
import ml.config as C

for f in sorted(os.listdir(C.FIG_DIR)):
    if f.endswith(".png"):
        print("\n" + "=" * 78 + "\n" + f)
        display(Image(filename=os.path.join(C.FIG_DIR, f)))

## 5. Key tables

In [ ]:
import pandas as pd
import ml.config as C

pd.set_option("display.width", 160, "display.max_columns", 40)
for t in ["t06_kmeans_sweep", "t14_cluster_summary", "t13_cluster_item_zscores",
          "t15_discriminating_items", "t10_theme_prevalence", "t09_algorithm_agreement"]:
    p = os.path.join(C.TAB_DIR, t + ".csv")
    if os.path.exists(p):
        print("\n" + "=" * 78 + "\n" + t)
        display(pd.read_csv(p, index_col=0))

## 6. Using the trained model

The pipeline persists a self-contained bundle. This is what makes the analysis a
*reusable instrument* rather than a one-off: next semester's responses can be
scored against this semester's profiles without re-fitting anything.

In [ ]:
import joblib, numpy as np, pandas as pd
import ml.config as C

bundle = joblib.load(os.path.join(C.MODEL_DIR, "stress_profile_model.joblib"))
print("k = %d | silhouette = %.4f | trained on n = %d"
      % (bundle["k"], bundle["silhouette"], bundle["trained_on"]["n"]))
for i, nm in bundle["cluster_names"].items():
    print("  cluster %d -> %s" % (i, nm))


def assign_profile(responses):
    """Score new responses.

    `responses` is a DataFrame with the twelve RAW item columns (1-5) named by the
    short keys in `bundle["items"]`. Reverse coding is applied here, so callers
    pass raw questionnaire answers exactly as exported.
    """
    X = responses[bundle["items"]].astype(float).copy()
    for c in bundle["reversed_items"]:
        X[c] = 6 - X[c]
    Z = bundle["scaler"].transform(X.to_numpy())
    labels = bundle["kmeans"].predict(Z)
    return pd.DataFrame({
        "cluster": labels,
        "profile": [bundle["cluster_names"][int(l)] for l in labels],
    }, index=responses.index)


demo = pd.DataFrame([
    dict(zip(bundle["items"], [5, 5, 5, 5, 5, 5, 5, 1, 1, 5, 5, 5])),   # high strain, no support
    dict(zip(bundle["items"], [2, 2, 2, 2, 2, 2, 2, 5, 5, 2, 2, 2])),   # low strain, well supported
])
display(assign_profile(demo))

## 7. Optional - multilingual embedding cross-check (GPU + internet)

The methodology report *asserts* that transformer embeddings are unreliable on
answers this short and this code-mixed. On Kaggle that claim can be **tested**
rather than assumed.

To run it: **Settings -> Accelerator: GPU**, **Settings -> Internet: On**, then run
the two cells below. If either is unavailable the pipeline records a skip and
nothing downstream changes.

In [ ]:
# !pip install -q sentence-transformers

In [ ]:
import ml.embeddings as EMB
import ml.lexicon as LX
import ml.dataio as IO
import ml.config as C

if EMB.available():
    df, _ = IO.load_raw()
    names = IO.column_names(df)
    txt = df[names["open_current"]]
    T, mask, _ = LX.tag_frame(txt)
    out = EMB.run(txt, mask, T)
    for k, v in out.items():
        print("%-28s %s" % (k, v))
else:
    print("sentence-transformers not installed - uncomment the pip cell above "
          "(needs Internet: On).")

## 8. Output manifest

In [ ]:
import ml.config as C

total = 0
for root, _, files in os.walk(C.OUT_DIR):
    if ".ipynb_checkpoints" in root or root.endswith("ml"):
        continue
    for f in sorted(files):
        p = os.path.join(root, f)
        sz = os.path.getsize(p)
        total += sz
        print("%9.1f KB  %s" % (sz / 1024, os.path.relpath(p, C.OUT_DIR)))
print("\n%.1f MB total" % (total / 1024 / 1024))